## Configure

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from os.path import join
from pathlib import Path
import yaml
from yaml.loader import SafeLoader
import pandas as pd
import geopandas as gpd
from shapely.geometry import box
import numpy as np
import rioxarray as rio
from scipy import stats
from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm


import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FuncFormatter
import matplotlib.ticker as mtick
import matplotlib.colors as colors
from matplotlib.colorbar import ColorbarBase
from mpl_toolkits.axes_grid1 import make_axes_locatable
import seaborn as sns
import contextily as cx
from pyproj import Transformer

import unsafe.files as unfile
import unsafe.exp as unexp

In [ ]:
# Name the fips, statefips, stateabbr, and nation that
# we are using for this analysis
fips_args = {
    'FIPS': ['42101'], 
    'STATEFIPS': ['42'],
    'STATEABBR': ['PA'],
    'NATION': ['US']
}
FIPS = fips_args['FIPS'][0]
NATION = fips_args['NATION'][0]

ABS_DIR = Path().absolute().parents[0]

CONFIG_FILEP = join(ABS_DIR, 'config', 'config.yaml')
# Open the config file and load
with open(CONFIG_FILEP) as f:
    CONFIG = yaml.load(f, Loader=SafeLoader)

# Wildcards for urls
URL_WILDCARDS = CONFIG['url_wildcards']

# Get the file extensions for api endpoints
API_EXT = CONFIG['api_ext']

# Get the CRS constants
NSI_CRS = CONFIG['nsi_crs']

# Dictionary of ref_names
REF_NAMES_DICT = CONFIG['ref_names']

# Dictionary of ref_id_names
REF_ID_NAMES_DICT = CONFIG['ref_id_names']

# Coefficient of variation
# for structure values
COEF_VARIATION = CONFIG['coef_var']

# First floor elevation dictionary
FFE_DICT = CONFIG['ffe_dict']

# Number of states of the world
N_SOW = CONFIG['sows']

# Data for flood depth grids
# Get hazard model variables
HAZ_FILEN = CONFIG['haz_filename']
# Get CRS for depth grids
HAZ_CRS = CONFIG['haz_crs']
# Ensemble members
HAZ_NENS = CONFIG['haz_nens']
# Number of columns for each depth grid
HAZ_NCOLS = CONFIG['haz_ncols']
# Num rows for each depth grid
HAZ_NROWS = CONFIG['haz_nrows']
# Lower left x coordinate
HAZ_XLL = CONFIG['haz_xll']
# Lower left y coordinate
HAZ_YLL = CONFIG['haz_yll']
# Cell resolution
HAZ_RES = CONFIG['haz_res']
# NODATA values
HAZ_NODATA = CONFIG['haz_nodata']

# Get the files we need downloaded
DOWNLOAD = pd.json_normalize(CONFIG['download'], sep='_').T

# We can also specify the filepath to the
# raw data directory
FR = join(ABS_DIR, "data", "raw")

# And external - where our hazard data should be
FE = join(FR, "external")

# Set up interim and results directories as well
# We already use "FR" for raw, we use "FO" 
# because you can also think of results
# as output
FI = join(ABS_DIR, "data", "interim")
FO = join(ABS_DIR, "data", "results")

# For figures
FIG_DIR = join(ABS_DIR, "fig")

# "Raw" data directories for exposure, vulnerability (vuln) and
# administrative reference files
EXP_DIR_R = join(FR, "exp")
VULN_DIR_R = join(FR, "vuln")
REF_DIR_R = join(FR, "ref")
# Haz is for depth grids
HAZ_DIR_R = join(FE, "haz")
# Pol is for NFHL
POL_DIR_R = join(FR, "pol")

# Unzip directory 
UNZIP_DIR = join(FR, "unzipped")

# We want to process unzipped data and move it
# to the interim directory where we keep
# processed data
# Get the filepaths for unzipped data
# We unzipped the depth grids (haz) and 
# ddfs (vuln) into the "external"/ subdirectory
HAZ_DIR_UZ = join(UNZIP_DIR, "external", "haz")
POL_DIR_UZ = join(UNZIP_DIR, "pol")
REF_DIR_UZ = join(UNZIP_DIR, "ref")
VULN_DIR_UZ = join(UNZIP_DIR, "external", "vuln")

# Our study domain
CLIP_SHP_FILEP = join(HAZ_DIR_UZ, 'RIFT_domain', 'domain_1.shp')

# "Interim" data directories
EXP_DIR_I = join(FI, "exp")
VULN_DIR_I = join(FI, "vuln")
REF_DIR_I = join(FI, "ref")
# Haz is for depth grids
HAZ_DIR_I = join(FI, "haz")
# Pol is for NFHL
POL_DIR_I = join(FI, "pol")

In [ ]:
unfile.prepare_saving(join(FIG_DIR, 'fig.1png'))

## Load data for analysis and plotting

In [ ]:
# Helpful to store the dataframes as a dictionary where the 
# experiment name is the key
# The naming conventions are different for considering uncertainty
# and this code accounts for that
output_dir = join(FO, 'ensembles')
dg_id = "009"
exps = ['phil', 'nsi_ddfs', 'nsi_unsafe',
        'nsi_phil', 'nsiadj_unsafe', 'nsi_allphil']

all_results = {}
for exp_name in exps:
    main_exp_name = f"main_exp_{dg_id}_{exp_name}_main.pqt"
    ens_result = pd.read_parquet(join(output_dir, main_exp_name))
    all_results[exp_name] = ens_result

nounc_name = f"no_unc_{dg_id}_main.pqt"
all_results["no_unc"] = pd.read_parquet(join(output_dir, nounc_name))

In [ ]:
# Spatial data
clip_geo = gpd.read_file(CLIP_SHP_FILEP)
tract_ref = gpd.read_file(join(REF_DIR_I, FIPS, 'tract.gpkg'))[['GEOID', 'geometry']]

# Inventories
# Spatial
nsi_clip_out = gpd.read_file(join(EXP_DIR_I, FIPS, 'nsi_res.gpkg'))
phil_inv_out = gpd.read_file(join(EXP_DIR_I, FIPS, 'phil_res.gpkg'))
# Ensembles
phil_inv_ens = pd.read_parquet(join(EXP_DIR_I, FIPS, 'phil_inv_ens.pqt'))
nsi_inv_ens = pd.read_parquet(join(EXP_DIR_I, FIPS, 'nsi_inv_ens.pqt'))

# Data to link inventories to tracts
phil_refs = pd.read_parquet(join(EXP_DIR_I, FIPS, 'phil_ref.pqt'))
nsi_refs = pd.read_parquet(join(EXP_DIR_I, FIPS, 'nsi_ref.pqt'))

# Depths across inundation model ensembles
nsi_depths_df = pd.read_parquet(join(EXP_DIR_I, FIPS, 'nsi_depths_updated.pqt'))
phil_depths_df = pd.read_parquet(join(EXP_DIR_I, FIPS, 'phil_depths.pqt')).set_index('bfid')

# Load in Philadelphia parcels linked to assessment records
# for investigating NSI records w/o a Philly res location match
phil_sub = gpd.read_file(join(EXP_DIR_I, FIPS, 'phil_all.gpkg'))
# Load in all NSI records for investigating Philly inventory records
# without a NSI match
nsi_gdf = unexp.get_nsi_geo(FIPS, NSI_CRS, EXP_DIR_R)
# Load parcels for linking NSI records to 
# Philly inventory records without a bld_fp match
parcel = gpd.read_file(join(EXP_DIR_R, FIPS, 'parcel.geojson'),
                       mask=clip_geo)


## Postprocessing results for figures

There are a few datasets that will be helpful for analyzing and visualizing the results. 

First, we want to identify consistent records between the structure inventories and categorize records in terms of their match types across inventories. This is the basis for several of our main results. For each match type, we want to know about damage discrepancies, rank discrepancies, etc., so there is a lot of processing here. 

Second, we want to have results across all experiments summarized at the census tract level. A main inquiry of research is whether damage discrepancies at the property level cancel out at the scales most typical for using prospective damage estimates in support of decisions. 

Finally, we want to calculate how well the various experiments compare to the "best" representation of damages using the Philly inventory. 

In [ ]:
# Can only choose depth grids that we generated ensembles for
dg_id = '009'
dam_col = 'naccs_loss_' + dg_id

### Get match types

We calculate match types across inventories. There are a few high level categories. First, we have NSI records & Philly records that uniquely link to each other over space. There are different levels of characteristic matching there. Second, we have  NSI records without a link to a Philly record over space. Third, there is the same as #2 except reversed. Finally, there are cases where several NSI records link to the same Philly record over space (many-to-one), which is problematic. We could arbitrarily take one of the links as a one-to-one and leave the rest as NSI only, but that is a different logic than the "true" NSI only cases. The "right" way to process these is probably to aggregate them (apartment buildings or condos most likely) but the issue is that when you only have the NSI data you don't know how to aggregate things like stories because do we know if the records are vertically stacked or side by side (and what if foundation type is different?). For loss-estimation purposes, we use what info we have for the DDFs. So, for post-proccessing, we do want to aggregate the NSI records that share a bfid link at least in terms of their loss difference. So, I think we should treat it as its own match type. The Philly properties will be there as well (because we're seeing how the losses net)

In [ ]:
# Spatial links between NSI and parcels
# in order to merge in the Philly building characteristics

# First, we need to keep track of how many buildings are linked
# with each parcel. Usually one to one, but we did have to do some
# disaggregation for condos & apartments
parcel_bld_cnt = (
    phil_inv_out
    .groupby('parcel_number')['bfid']
    .nunique()
    .rename('n_bldgs')
)

phil_inv_out = phil_inv_out.merge(
    parcel_bld_cnt,
    left_on='parcel_number',
    right_index=True,
    how='left'
)

# Next, join all NSI records to building footprints
# with a spatial merge that we can
lnk_nsi_bld = gpd.sjoin(nsi_clip_out,
                        phil_inv_out[['bfid', 'parcel_number', 'n_bldgs', 'geometry']],
                        how='inner',
                        predicate='intersects')
lnk_nsi_bld['lnk_src'] = 'bld_fp'

# We want to keep track of the fd_id and bfid that have been connected this way
matched_fd_direct = lnk_nsi_bld['fd_id'].unique()
matched_bfid_direct = lnk_nsi_bld['bfid'].unique()


# Now we remove all directly matched records from nsi_clip_out and
# phil_inv_out. We do this so that when we do a fallback merge of 
# NSI on parcels, we are not introducing any duplications
nsi_unmatched = nsi_clip_out[
    ~nsi_clip_out['fd_id'].isin(matched_fd_direct)
]
phil_unmatched = phil_inv_out[
    ~phil_inv_out['bfid'].isin(matched_bfid_direct)
]

# Now we do the spatial merge of NSI points within parcels
# We don't subset parcels to parcel_number not in lnk_nsi_bld
# because there can be many buildings to one parcel
lnk_nsi_parcels = gpd.sjoin(nsi_unmatched,
                            parcel[['BRT_ID', 'geometry']],
                            how='inner',
                            predicate='intersects')

# Now we connect these NSI records, which have a BRT_ID link,
# to the unmatched Philly records
nsi_par_links = lnk_nsi_parcels[['fd_id', 'BRT_ID']].merge(phil_unmatched[['parcel_number', 'bfid', 'n_bldgs']],
                                                            left_on='BRT_ID',
                                                            right_on='parcel_number',
                                                            how='inner')

# We need to distinguish between NSI records linked to Philly records
# with 1 building, and those with more than 1
single_bld_links = nsi_par_links[nsi_par_links['n_bldgs'] == 1].copy()
multi_bld_links = nsi_par_links[nsi_par_links['n_bldgs'] > 1].copy()

# When there is a single link, we know we are matching up NSI & Philly building
# that "should" go together, but the location is a bit off. This will generally
# be very minor (esp. relative to FFE uncertainty),  so we want to process these 
# as characteristic mismatches (but will treat them as location mismatches for
# some supplementary analyses)
fallback_links = single_bld_links[['fd_id', 'bfid', 'n_bldgs', 'parcel_number']].copy()
fallback_links['lnk_src'] = 'parcel'

# When there isn't, we're not realy sure what building to assign as a match
# to the NSI record. These NSI records are a type of NSI only and these
# Philly records are a type of Philly only. What we'll do is
# create mismatch & mismatch reason columns to the subset
# We can take stock of the bfid before that and make sure we pull those out
# of phil_unmatched and set their mismatch_reason & mismatch appropriately

ambiguous_nsi = multi_bld_links['fd_id'].unique()
ambiguous_phil = multi_bld_links['bfid'].unique()

ambiguous_nsi_df = (
    nsi_unmatched[nsi_unmatched['fd_id'].isin(nsi_inv_ens.index)]
    .merge(
        multi_bld_links[['fd_id', 'parcel_number']].drop_duplicates(),
        on='fd_id',
        how='inner'
    )
)
ambiguous_phil_df = (
    phil_unmatched[phil_unmatched['bfid'].isin(phil_inv_ens.index)]
    .merge(
        multi_bld_links[['bfid', 'parcel_number']].drop_duplicates(),
        on=['bfid', 'parcel_number'],
        how='inner'
    )
)

ambiguous_nsi_df['mismatch'] = 'NSI Only: Location Imprecision'
ambiguous_nsi_df['mismatch_reason'] = 'Many to One Parcel, No Building Link'
ambiguous_phil_df['mismatch'] = 'Philly Only: Location Imprecision'
ambiguous_phil_df['mismatch_reason'] = 'Many to One Parcel, No NSI Link'

# Prepare these dataframes for later concatenations
ambiguous_nsi_df = ambiguous_nsi_df.rename(
    columns={
        'val_struct': 'val_struct_nsi',
        'num_story': 'num_story_nsi',
        'found_type': 'found_type_nsi',
        'occtype': 'occtype_nsi'
    }
)
ambiguous_nsi_sub = ambiguous_nsi_df[
    [
        'fd_id',
        'parcel_number',
        'mismatch',
        'mismatch_reason',
        'val_struct_nsi',
        'found_ht',
        'source',
        'ftprntsrc'
    ]
].copy()

ambiguous_phil_df = ambiguous_phil_df.rename(
    columns={
        'val_struct': 'val_struct_phil',
        'occ_type': 'occtype_phil',
        'b_type': 'found_type_phil',
        'ddf_stories': 'num_story_phil'
    }
)
ambiguous_phil_sub = ambiguous_phil_df[
    [
        'bfid',
        'parcel_number',
        'mismatch',
        'mismatch_reason',
        'val_struct_phil',
        'occtype_phil',
        'stories_n',
        'found_type_phil',
        'num_story_phil'
    ]
].copy()

# Update NSI & Phil unmatched based on latest subsets
nsi_only = nsi_unmatched[(~nsi_unmatched['fd_id'].isin(fallback_links['fd_id'])) &
                         (~nsi_unmatched['fd_id'].isin(ambiguous_nsi))]
philly_only = phil_unmatched[(~phil_unmatched['bfid'].isin(fallback_links['bfid'])) &
                             (~phil_unmatched['bfid'].isin(ambiguous_phil))]

# Combine the matched with fallbacks
nsi_phil_matched = pd.concat([lnk_nsi_bld[['fd_id', 'bfid', 'parcel_number', 'n_bldgs', 'lnk_src']],
                              fallback_links[['fd_id', 'bfid', 'parcel_number', 'n_bldgs', 'lnk_src']]],
                              axis=0,
                              ignore_index=True)

# Subset philly_only to entries in phil_inv_ens
# Use structure characteristics from phil_inv_ens
p_cols = ['bfid', 'parcel_number', 'geometry']
philly_only = philly_only.loc[:, p_cols].copy()
philly_only = philly_only.merge(phil_inv_ens.reset_index(),
                                on='bfid')


Now that we have the high-level match types, we will process low-level types of mismatches, starting with `nsi_phil_matched`, which includes all links between Philly & NSI residences based on spatial merges. This high-level match will include instances where structure characteristics disagree, subsets disagree (e.g., NSI inaccurately classifies a > or <= 3 story res) , or everything matches. 

In [ ]:
matched = (
    nsi_phil_matched
    .merge(
        phil_inv_out[['bfid', 'stories_n']],
        on='bfid',
        how='left'
    )
    .merge(
        nsi_clip_out[['fd_id']],
        on='fd_id',
        how='left',
        suffixes=('_phil', '_nsi')
    )
)

matched['in_philly_sub'] = (
    matched['bfid']
    .isin(phil_inv_ens.index)
)

matched['in_nsi_sub'] = (
    matched['fd_id']
    .isin(nsi_inv_ens.index)
)

# Drop all records that aren't in either
matched = matched[(matched['in_philly_sub']) | 
                  (matched['in_nsi_sub'])].copy()

# If the structure is not in the NSI inventory, but is in Phil,
# that means NSI thinks the structure has 4+ stories
matched['nsi_stories_over'] = matched['in_nsi_sub'] == False
# Vice versa, if the structure is not in the Philly inventory, but is in NSI
matched['nsi_stories_under'] = matched['in_philly_sub'] == False

# If either of these conditions are met, this is a type of NSI Only
# A more detailed level of mismatch can be called 'Low-Rise Misclassification'
matched.loc[(matched['nsi_stories_over']),
            'mismatch_reason'] = 'NSI Overestimates Stories'
matched.loc[(matched['nsi_stories_under']),
            'mismatch_reason'] = 'NSI Underestimates Stories'

# Add higher-level mismatch column
matched.loc[(matched['nsi_stories_over']),
            'mismatch'] = 'Philly Only: Low-Rise Misclassification'
matched.loc[(matched['nsi_stories_under']),
            'mismatch'] = 'NSI Only: Low-Rise Misclassification'

# Next, we check for DDF mismatches based on stories & foundation
matched = (
    matched
    .merge(
        phil_inv_ens.reset_index()[[
            'bfid',
            'num_story',
            'found_type',
            'val_struct',
            'occtype'
        ]],
        on='bfid',
        how='left'
    )
    .merge(
        nsi_inv_ens.reset_index()[[
            'fd_id',
            'num_story',
            'found_type',
            'val_struct',
            'occtype'
        ]],
        on='fd_id',
        how='left',
        suffixes=('_phil', '_nsi')
    )
)

lowrise_sub = matched.loc[
    matched['mismatch_reason'].notnull()
].copy()

# Only want to assess DDF mismatch for the records that are in both inv_ens dfs
ddf_sub = matched.loc[
    matched['mismatch_reason'].isnull()
].copy()

# This helps us do the mismatch types with less code
def _get_ddf(stories, found_type):

    if pd.isnull(stories) or pd.isnull(found_type):
        return np.nan

    found = {
        'B': 'WB',
        'S': 'NB'
    }.get(found_type)

    if found is None:
        return np.nan

    return f'{int(stories)}S{found}'

ddf_sub['nsi_ddf'] = ddf_sub.apply(
    lambda x: _get_ddf(x['num_story_nsi'],
                      x['found_type_nsi']),
    axis=1
)

ddf_sub['phil_ddf'] = ddf_sub.apply(
    lambda x: _get_ddf(x['num_story_phil'],
                      x['found_type_phil']),
    axis=1
)

# Check if DDFs match per linked record
ddf_sub['ddf_match'] = (
    ddf_sub['nsi_ddf'] ==
    ddf_sub['phil_ddf']
)

# Track DDF matches & mismatches
ddf_sub['ddf_comp'] = (
    'NSI: ' +
    ddf_sub['nsi_ddf'] +
    '\nPhilly: ' +
    ddf_sub['phil_ddf']
)
# When match, simplify name
ddf_sub.loc[ddf_sub['ddf_match'],
            'ddf_comp'] = 'Both ' + ddf_sub['phil_ddf']

# Get a mismatch reason for DDF mismatches and then
# merge back into matched df
# Detailed mismatch reason
ddf_sub['mismatch_reason'] = 'None'

# Only add a mismatch reason where DDFs differ
ddf_sub.loc[~ddf_sub['ddf_match'],
            'mismatch_reason' ] = ddf_sub['ddf_comp']

# Higher-level mismatch category
ddf_sub['mismatch'] = 'DDF Match'
ddf_sub.loc[ddf_sub['mismatch_reason'] != 'None',
            'mismatch'] = 'DDF Mismatch'

# Concat back together
matched = pd.concat(
    [lowrise_sub, ddf_sub],
    ignore_index=True
)

Now we will work with the NSI only and Philly only dataframes. `phil_sub` includes non residential Philly parcels that can be matched to NSI records not linked to nsi_clip_out. We want to classify 'NSI Only' in more detail. It's actually a combo of NSI records linked to non-residential structures/parcels, and NSI records linked to vacant land (and possibly in water bodies). We will do the same for properties labeled as 'Unmatched Philly Res' using the `nsi_gdf` dataframe (all NSI structures). We can likely connect these to properties in the NSI, but not in the subset for our analysis. 

In [ ]:
nsi_loc_check = gpd.sjoin(nsi_only[nsi_only['fd_id'].isin(nsi_inv_ens.index)],
                          phil_sub[['parcel_number', 'building_code_description', 'geometry',
                                    'category_code', 'category_code_description']],
                                    predicate='within',
                                    how='inner') 
# For properties w/ category_code 1, 2, or 14, these are the "correct" categories 
# But if there's no building footprint, they are vacant
# Codes 6, 12, and 13 are vacant land 
# Others are the wrong types of buildings, such as commercial  
# For Unmatched Philly, we want to take the phil_only dataframe and find nsi_gdf points 
# that link up to records with a spatial join. Then we can see what kind of NSI records 
# these are, or whether NSI misses them.
nsi_loc_check.loc[
    nsi_loc_check['parcel_number'].isnull(),
    'mismatch_reason'
] = 'No Parcel Match'

vacant_codes = ['6', '12', '13']

nsi_loc_check.loc[
    nsi_loc_check['category_code'].str.strip().isin(vacant_codes),
    'mismatch_reason'
] = 'Vacant Parcel'

res_codes = ['1', '2', '14']

nsi_loc_check.loc[
    (~nsi_loc_check['category_code'].str.strip().isin(res_codes)) &
    (~nsi_loc_check['category_code'].str.strip().isin(vacant_codes)) &
    (nsi_loc_check['mismatch_reason'].isnull()),
    'mismatch_reason'
] = 'Non-Res. Parcel'

nsi_loc_check.loc[
    nsi_loc_check['category_code'].str.strip().isin(res_codes) &
    (nsi_loc_check['mismatch_reason'].isnull()),
    'mismatch_reason'
] = 'Vacant Parcel'

# Note that Vacant Parcel and Res. Parcel w/o Structure
# are the same thing but with different logic. The vacant parcel
# is a code in the assessor data and from random parcel cross-checks
# seems accurate. The Res. Parcel w/o Structure are cases where the NSI point falls within 
# a residential parcel, but there is no structure in the Philly inventory on that parcel.
# Presumably these will be labeled vacant parcels eventually. 
nsi_loc_check['mismatch'] = 'NSI Only: Vacant or Non-Res.'

In [ ]:
# Some of these are in parcels that fall outside of the study boundary
# Some of these are in parcels not associated with tax assessments
# This could mean that the parcel is subdivided and hasn't been assessed yet
# In these cases, I haven't seen an instance of two Philly footprints, 
# so it's essentially a minor location issue. So, overall these
# are two kinds of NSI Only records, both location-based. 
# One is location outside of study boundary, and other is
# location offset. 


# Remaining NSI-only records
nsi_only_loc = nsi_only[
    (nsi_only['fd_id'].isin(nsi_inv_ens.index)) &
    (~nsi_only['fd_id'].isin(nsi_loc_check['fd_id']))
].copy()

# Join to parcels to delineate between those oustide study area and in
nsi_only_loc = gpd.sjoin(
    nsi_only_loc,
    parcel,
    predicate='within',
    how='left'
)

# High-level mismatch
nsi_only_loc['mismatch'] = 'NSI Only: Location Mismatch'

# If the join didn't find a parcel, it's either
# in the study area and not in a parcel
# or outside of the study area (in or out of a parcel)
nsi_only_loc.loc[
    nsi_only_loc['index_right'].isnull(),
    'mismatch_reason'
] = 'No Parcel Match'

# If the join found a parcel but the parcel lacks BRT_ID
# this is a parcel without an assessment or footprint
# It should be safe to drop these to avoid unfair comparisons 
nsi_only_loc.loc[
    (nsi_only_loc['index_right'].notnull()) &
    (nsi_only_loc['BRT_ID'].isnull()),
    'mismatch_reason'
] = 'Parcel Without Assessment'

# If there is a BRT_ID, it refers to a tax parcel
# with an assessment or out of study area footprint
# so we should drop these to avoid unfair comparisons
nsi_only_loc.loc[
    nsi_only_loc['BRT_ID'].notnull(),
    'mismatch_reason'
] = 'Parcel Without Building'

# Combine NSI-only records retained for analysis
# Remember to drop the SA ones for main analysis
nsi_only_full = pd.concat(
    [nsi_loc_check, nsi_only_loc],
    ignore_index=True,
    axis=0
)

And now figuring out what the NSI thinks about these Philly Only structures

In [ ]:
# Spatial join for the building footprints and NSI points
philly_only_loc_check = gpd.sjoin(philly_only,
                                  nsi_gdf[['fd_id', 'occtype', 'num_story', 'source', 'geometry']],
                                  predicate='contains',
                                  lsuffix='phil',
                                  rsuffix='nsi',
                                  how='inner')

# Update columns in philly_only to have the suffix (not merging in all from nsi)
philly_only_loc_check = philly_only_loc_check.rename(columns={'val_struct': 'val_struct_phil',
                                                              'found_type': 'found_type_phil'})

# These are structures where the NSI points fall outside of our study region clip
# or the NSI classifies the residential structures as non-residential
# This is a location imprecision issue
# There are a few RES4 instances and it turns out these are accurate - these are labeled as
# hotels/rooming house in the Philly inventory. So although we orig. labeled these as RES4, it would
# it would be unfair to keep these in for damage comparison purposes. We should drop these
# from philly_inv_ens. For those without RES4, though, we can retain them as
# a location mismatch. We can call it 'NSI Low-Rise Outside Study Region'


# Subset to NSI RES4 records in philly_only_loc_check and store these bfid
# to drop from nsi_inv_ens before producing any results 
drop_phil_ids = philly_only_loc_check.loc[philly_only_loc_check['occtype_nsi'].str.contains('RES4')]
drop_phil_ids = drop_phil_ids['bfid'].unique()


# For remainder of records in philly_only_loc_check
# label mismatch reason as 'NSI Low-Rise Outside Study Region' if RES* NSI occtype
# otherwise label as NSI Non-Res. 
# Mismatch is Philly Only
philly_only_loc = philly_only_loc_check[~philly_only_loc_check['bfid'].isin(drop_phil_ids)].copy()

philly_only_loc.loc[
    philly_only_loc['occtype_nsi'].str.contains('RES'),
    'mismatch_reason'
] = 'NSI Low-Rise Outside Study Region'

philly_only_loc.loc[
    ~philly_only_loc['occtype_nsi'].str.contains('RES'),
    'mismatch_reason'
] = 'NSI Non-Res.'

philly_only_loc['mismatch'] = 'Philly Only: Low-Rise Misclassification'

# There are also Philly Only structures that don't have a NSI record in them
philly_only_missing = philly_only[
    ~philly_only['bfid'].isin(philly_only_loc_check['bfid'])
].copy()

philly_only_missing['mismatch'] = 'Philly Only: No NSI Structure'
philly_only_missing['mismatch_reason'] = 'No NSI Structure'

philly_only_missing = philly_only_missing.rename(columns={'occtype': 'occtype_phil',
                                                          'num_story': 'num_story_phil',
                                                          'val_struct': 'val_struct_phil',
                                                          'found_type': 'found_type_phil'})

# Can concat these
philly_only_full = pd.concat([philly_only_loc, philly_only_missing],
                             ignore_index=True, axis=0)

Now concat these - after appropriate subsets - so that we have the dataframe of NSI & Philly records (linked as appropriate) for merging in loss estimates. 

In [ ]:
# Dataframe of all location connected NSI & Philly records
# Some are instances of NSI misclassifying a low rise (either over or undercounting stories)
# Some are instances of DDFs mismatching
# Some are instances of DDFs matching

# Dataframe of all NSI records that link to a vacant parcel or non-res parcel
# nsi_loc_check
# Subset to columns of interest before concat
nsi_loc_cols= ['fd_id', 'mismatch_reason', 'mismatch',
               'category_code_description', 'found_ht', 'source',
               'ftprntsrc', 'parcel_number']
nsi_app_cols = ['val_struct', 'num_story', 'found_type', 'occtype']
nsi_only_sub = nsi_only_full.loc[:, nsi_loc_cols].merge(nsi_inv_ens.loc[:, nsi_app_cols].reset_index(),
                                                        on='fd_id')
# For each column in nsi_app_cols, add '_nsi' suffix to match with matched df for concat later
nsi_only_sub = nsi_only_sub.rename(columns={col: col + '_nsi' for col in nsi_app_cols})

# Dataframe of all Philly records that link to a NSI record that
# falls outside of the study region clip or Philly records
# that have no NSI record at all in them
# philly_only_full
philly_only_drop_cols = ['geometry', 'tract_id']
phil_only_sub = philly_only_full.drop(columns=philly_only_drop_cols).copy()

matched_df = pd.concat([matched, nsi_only_sub, phil_only_sub,
                        ambiguous_nsi_sub, ambiguous_phil_sub],
                       ignore_index=True, axis=0)

# Note: we want a list of fd_ids to drop for the SA of NSI Only
drop_nsi_ids = nsi_only_loc.loc[
    nsi_only_loc['mismatch_reason'].isin([
        'Parcel Without Assessment',
        'Parcel Without Building'
    ]),
    'fd_id'
].unique()

# Matched_df_main will drop these
matched_df_main = matched_df[
    ~matched_df['fd_id'].isin(drop_nsi_ids)
]

In [ ]:
# Checking that there are 12 fewer Philly records (from RES4 check)
# in various mismatches than in philly_inv_ens
# matched_df_main.drop_duplicates('bfid', keep='first')['mismatch'].value_counts()
# mismatch
# DDF Mismatch                               56014
# DDF Match                                  34106
# NSI Only: Low-Rise Misclassification        1468
# Philly Only: Low-Rise Misclassification     1349
# Philly Only: No NSI Structure                397
# Philly Only: Location Imprecision             39
# NSI Only: Vacant or Non-Res.                   1
# The sum of Philly records above should be 91905
# 56014+34106+1349+397+39 = 91905, so checks out

# Checking that there are the same number of NSI records
# in various mismatches as nsi_inv_ens, after dropping the drop_nsi_ids
# matched_df_main.drop_duplicates('fd_id', keep='first')['mismatch'].value_counts()
# mismatch
# DDF Mismatch                               56935
# DDF Match                                  34284
# NSI Only: Low-Rise Misclassification        1850
# Philly Only: Low-Rise Misclassification     1379
# NSI Only: Vacant or Non-Res.                1114
# NSI Only: Location Mismatch                  201
# NSI Only: Location Imprecision                43
# Philly Only: No NSI Structure                  1
# 56935+34284+1850+1114+201+43 = 94427
# len(nsi_inv_ens[~nsi_inv_ens.index.isin(drop_nsi_ids)]) = 94427


In [ ]:
phil_matches = ['DDF Mismatch', 'DDF Match', 'Philly Only: No NSI Structure',
                'Philly Only: Low-Rise Misclassification',
                'Philly Only: Location Imprecision']
matched_df_main[matched_df_main['mismatch'].isin(phil_matches)].drop_duplicates('bfid', keep='first')['occtype_phil'].value_counts()/91905

In [ ]:
nsi_matches = ['DDF Mismatch', 'DDF Match', 'NSI Only: Low-Rise Misclassification',
               'NSI Only: Vacant or Non-Res.', 'NSI Only: Location Mismatch',
               'NSI Only: Location Imprecision']
matched_df_main[matched_df_main['mismatch'].isin(nsi_matches)].drop_duplicates('fd_id', keep='first')['occtype_nsi'].value_counts()/94427

The `matched_df_main` dataframe contains all the records we need for comparing structure-level losses. We can load in the loss dataframes and depths and merge these into the matched dataframe. Then we can groupby on the `mismatch` column for figures as needed. For supplementary figures and summary statistics, we can look at `mismatch_reason`. In our case study, we are using the ensemble to estimate "best guess" damage estimates with the Philly inventory, so we are taking the discrepancy relative to the *mean*. 

In [ ]:
# Create a df of the few match columns we're really interested in for plotting purposes
match_analysis = matched_df_main[['fd_id', 'bfid', 'parcel_number',
                                  'val_struct_nsi', 'val_struct_phil',
                                  'mismatch', 'mismatch_reason']].copy()

# Add parcel_number as a unique identifier
# Use fd_id for the structures from NSI that have no link
match_analysis['comp_id'] = (
    match_analysis['parcel_number']
    .fillna(match_analysis['fd_id'])
)

# We are going to groupby on comp_id after we merge in 
# ensemble generation results so that we can compare
# NSI/Philly estimates for more complex residential structures
# that are best characterized at the parcel level
# Most records are 1-1 NSI/Building/Parcel so this won't change much
# But we do want to note which parcels have been aggregated. We should
# keep track of how many fd_id & bfid have been aggregated, and update
# the mismatch category for these to reflect that they are aggregated and not 1-1 comparisons

# Combine the mean damage estimates from the ensemble
# of NSI fixed, DDF uncertain with the NSI inventory
nsi_ddf_unc = all_results['nsi_ddfs'].copy()
# All nan correspond to no estimated damages
nsi_ddf_mean = nsi_ddf_unc.groupby('fd_id')[[dam_col]].mean().fillna(0).reset_index()
# Merge depths in
nsi_ddf_mean[dg_id] = nsi_ddf_mean['fd_id'].map(nsi_depths_df[dg_id])*3.28084

# Prepare the columns (besides fd_id) for a merge for discrepancy calculations
nsi_analysis = nsi_ddf_mean.rename(
    columns={
        dam_col: dam_col + '_nsi',
        dg_id: dg_id + '_nsi'
    }
)

# Link loss & depth estimates with match details
match_analysis = match_analysis.merge(
    nsi_analysis,
    on='fd_id',
    how='left'
)

# Repeat for main Philadelphia estimates
main_phil = all_results['phil'].copy()
# Get relative loss
main_phil['rel_loss_phil'] = main_phil[dam_col] / main_phil['val_s']
# Groupby for mean damages
phil_mean = main_phil.groupby('bfid')[[dam_col, 'rel_loss_phil']].mean().reset_index()
# Merge in depths
phil_mean[dg_id] = phil_mean['bfid'].map(phil_depths_df[dg_id])*3.28084

# Rename for merge
phil_analysis = phil_mean.rename(
    columns={
        dam_col: dam_col + '_phil',
        dg_id: dg_id + '_phil'
    }
)
# Merge in Philly estimates for discrepancy calculations
match_analysis = match_analysis.merge(
    phil_analysis,
    on='bfid',
    how='left'
)

# Retain only rows with damage estimates
match_analysis = match_analysis[(match_analysis['009_nsi'].notnull()) |
                                (match_analysis['009_phil'].notnull())].copy()

# Create flags for whether the record has a loss
# We will sum these to indicate which parcels consist of multiple records
match_analysis['n_nsi'] = match_analysis['009_nsi'].notnull().astype(int)
match_analysis['n_phil'] = match_analysis['009_phil'].notnull().astype(int)

# Prepare a dummy bfid column for groupby
match_analysis['bfid_group'] = match_analysis['bfid'].fillna(-1)

# Add sq ft column
phil_area = dict(
    zip(
        phil_inv_out['bfid'],
        np.where(
            phil_inv_out['total_livable_area'].notnull(),
            phil_inv_out['total_livable_area'],
            phil_inv_out['total_area']
        )
    )
)
nsi_area = dict(
    zip(
        nsi_clip_out['fd_id'],
        nsi_clip_out['sqft']
    )
)

match_analysis['sqft_nsi'] = (
    match_analysis['fd_id']
    .map(nsi_area)
)

match_analysis['sqft_phil'] = (
    match_analysis['bfid']
    .map(phil_area)
)

# Add DDFs
phil_ddf = (
    phil_inv_ens['num_story']
    .astype(str)
    +
    np.where(
        phil_inv_ens['found_type'] == 'B',
        'SWB',
        'SNB'
    )
)

nsi_ddf = (
    nsi_inv_ens['num_story']
    .astype(str)
    +
    np.where(
        nsi_inv_ens['found_type'] == 'B',
        'SWB',
        'SNB'
    )
)

match_analysis['ddf_phil'] = (
    match_analysis['bfid']
    .map(phil_ddf)
)

match_analysis['ddf_nsi'] = (
    match_analysis['fd_id']
    .map(nsi_ddf)
)

# Aggregate such that any many NSI to one bld_fp gets grouped together
# We do not want to aggregate the Philly records in any way in this step
match_gb = match_analysis.groupby(['comp_id', 'bfid_group']).agg({
    'n_nsi': 'sum',
    'n_phil': 'first',
    'val_struct_nsi': 'sum',
    'val_struct_phil': 'first',
    dam_col + '_nsi': 'sum',
    dam_col + '_phil': 'first',
    'rel_loss_phil': 'first',
    dg_id + '_nsi': 'median',
    dg_id + '_phil': 'first',
    'mismatch': 'first',
    'mismatch_reason': 'first',
    'sqft_nsi': 'sum',
    'sqft_phil': 'first',
    'ddf_nsi': 'first',
    'ddf_phil': 'first'
}).reset_index()

# Calculate a modified loss for Philly based on rel_loss_phil
# and the aggregated NSI structure value
# When val_struct_nsi is NA, this defaults to the Philly loss estimate
match_gb[dam_col + '_phil_adj'] = (match_gb['rel_loss_phil'] * 
                                   match_gb['val_struct_nsi'])

# Aggregate for parcel-level matches. This is generally a 1:1 NSI/bld_fp match
# but it handles complex condo/apartment situations better
# The reason we don't groupby on comp_id directly before is that it would
# result in us double counting some Philly records
parcel_gb = match_gb.groupby(['comp_id']).agg({
    'n_nsi': 'sum',
    'n_phil': 'sum',
    'val_struct_nsi': 'sum',
    'val_struct_phil': 'sum',
    dam_col + '_nsi': 'sum',
    dam_col + '_phil': 'sum',
    dam_col + '_phil_adj': 'sum',
    dg_id + '_nsi': 'median',
    dg_id + '_phil': 'median',
    'mismatch': 'first',
    'mismatch_reason': 'first',
    'sqft_nsi': 'sum',
    'sqft_phil': 'mean',
    'ddf_nsi': 'first',
    'ddf_phil': 'first'
}).reset_index()

# Update mismatch & mismatch_reason for aggregated records
# Most will stay the same. Most are 1-1. Some are some-0, 0-some, which
# will have the correct mismatch assigned. 
# Anytime there is at least 1 in one, we will need to do an update
multi_mask = (
    (parcel_gb['n_nsi'] > 1) |
    (parcel_gb['n_phil'] > 1)
)

# In those cases, when the number of buildings is unequal
# the high-level mismatch category is Multi-Building Parcel Mismatch
parcel_gb.loc[
    multi_mask &
    (parcel_gb['n_nsi'] != parcel_gb['n_phil']),
    'mismatch'
] = 'Multi-Building Parcel Mismatch'
# We can add more detail based on which side has more buildings
parcel_gb.loc[
    multi_mask &
    (parcel_gb['n_nsi'] > parcel_gb['n_phil']),
    'mismatch_reason'
] = 'NSI More Structures'
parcel_gb.loc[
    multi_mask &
    (parcel_gb['n_phil'] > parcel_gb['n_nsi']),
    'mismatch_reason'
] = 'Philly More Structures'

# When multi & equal, we can just indicate that
parcel_gb.loc[
    multi_mask &
    (parcel_gb['n_nsi'] == parcel_gb['n_phil']),
    'mismatch'
] = 'Multi-Building Parcel Match'
parcel_gb.loc[
    multi_mask &
    (parcel_gb['n_nsi'] == parcel_gb['n_phil']),
    'mismatch_reason'
] = 'Equal Structures Count'

# Finally, when multi and n_phil = 0, 
# just treat as NSI Only
# misimatch_reason is NSI Underestimates Stories
parcel_gb.loc[
    multi_mask &
    (parcel_gb['n_phil'] == 0),
    'mismatch'
] = 'NSI Only: Low-Rise Misclassification'
parcel_gb.loc[
    multi_mask &
    (parcel_gb['n_phil'] == 0),
    'mismatch_reason'
] = 'NSI Underestimates Stories'


# Create parsimonious mismatch_plot column
parcel_gb['mismatch_plot'] = parcel_gb['mismatch']

parcel_gb.loc[
    parcel_gb['mismatch'].str.startswith(
        'NSI Only',
        na=False
    ),
    'mismatch_plot'
] = 'NSI Only'

parcel_gb.loc[
    parcel_gb['mismatch'].str.startswith(
        'Philly Only',
        na=False
    ),
    'mismatch_plot'
] = 'Philly Only'

parcel_gb.loc[
    parcel_gb['mismatch'].str.startswith(
        'Multi-Building',
        na=False
    ),
    'mismatch_plot'
] = 'Multiple Buildings\nOn Parcel'

# Add relative loss columns
parcel_gb['rel_loss_nsi'] = (
    parcel_gb[f'{dam_col}_nsi'] /
    parcel_gb['val_struct_nsi']
)

parcel_gb['rel_loss_phil'] = (
    parcel_gb[f'{dam_col}_phil'] /
    parcel_gb['val_struct_phil']
)

# Calculate discrepancies
parcel_gb['loss_diff'] = (
    parcel_gb[f'{dam_col}_nsi'].fillna(0) -
    parcel_gb[f'{dam_col}_phil'].fillna(0)
)

parcel_gb['loss_diff_adj'] = (
    parcel_gb[f'{dam_col}_nsi'] -
    parcel_gb[f'{dam_col}_phil_adj']
)

parcel_gb['value_diff'] = (
    parcel_gb['val_struct_nsi'].fillna(0) -
    parcel_gb['val_struct_phil'].fillna(0)
)

parcel_gb['rel_loss_diff'] = (
    parcel_gb['rel_loss_nsi'].fillna(0) -
    parcel_gb['rel_loss_phil'].fillna(0)
)

# Get absolute differences into thousands for plotting
parcel_gb['loss_diff_thou'] = (
    parcel_gb['loss_diff'] / 1e3
)

parcel_gb['val_diff_thou'] = (
    parcel_gb['value_diff'] / 1e3
)

# Get depth bins for plotting
parcel_gb['depth_ft'] = parcel_gb[f'{dg_id}_phil']

parcel_gb.loc[
    parcel_gb['depth_ft'].isnull(),
    'depth_ft'
] = parcel_gb[f'{dg_id}_nsi']

parcel_gb['depth_bins'] = pd.cut(
    parcel_gb['depth_ft'],
    bins=[0, 1, 2, 4, parcel_gb['depth_ft'].max()]
)

# Get val per sq ft
parcel_gb['val_sqft_nsi'] = (
    parcel_gb['val_struct_nsi']
    /
    parcel_gb['sqft_nsi']
)

parcel_gb['val_sqft_phil'] = (
    parcel_gb['val_struct_phil']
    /
    parcel_gb['sqft_phil']
)

# Get discrepancies for sq ft and val per sq ft
parcel_gb['sqft_diff'] = (
    parcel_gb['sqft_nsi']
    -
    parcel_gb['sqft_phil']
)

parcel_gb['val_sqft_diff'] = (
    parcel_gb['val_sqft_nsi']
    -
    parcel_gb['val_sqft_phil']
)

### Summaries of mismatches and discrepancies

In [ ]:
print('\n' + '='*80)
print('PARCEL-LEVEL MISMATCH COUNTS')
print('='*80)

print(
    parcel_gb
    .groupby('mismatch_plot')
    .size()
    .sort_values(ascending=False)
)

print('\n' + '='*80)
print('PARCEL-LEVEL MISMATCH PROPORTIONS')
print('='*80)

print(
    (
        parcel_gb
        .groupby('mismatch_plot')
        .size()
        / len(parcel_gb)
    )
    .sort_values(ascending=False)
    .round(4)
)

# STRUCTURE VALUE COMPARISON BY DDF TYPE

print('\n' + '='*80)
print('PHILADELPHIA INVENTORY')
print('Mean Structure Value and Structure Count by Stories/Foundation')
print('='*80)

print(
    phil_inv_ens[
        phil_inv_ens.index.isin(
            match_analysis['bfid']
        )
    ]
    .groupby(
        ['num_story', 'found_type']
    )
    .agg({
        'val_struct': [
            np.mean,
            np.size
        ]
    })
    .round(0)
)

print('\n' + '='*80)
print('NSI INVENTORY')
print('Mean Structure Value and Structure Count by Stories/Foundation')
print('='*80)

print(
    nsi_inv_ens[
        nsi_inv_ens.index.isin(
            match_analysis['fd_id']
        )
    ]
    .groupby(
        ['num_story', 'found_type']
    )
    .agg({
        'val_struct': [
            np.mean,
            np.size
        ]
    })
    .round(0)
)

# RELATIVE LOSS DISCREPANCIES

print('\n' + '='*80)
print('RELATIVE LOSS DISCREPANCY')
print('Grouped by Depth Bin and Mismatch Type')
print('='*80)

print(
    parcel_gb
    .groupby(
        ['depth_bins', 'mismatch_plot']
    )['rel_loss_diff']
    .describe()
    .round(4)
)

# DOLLAR LOSS DISCREPANCIES

print('\n' + '='*80)
print('DOLLAR LOSS DISCREPANCY ($)')
print('Grouped by Depth Bin and Mismatch Type')
print('='*80)

print(
    parcel_gb
    .groupby(
        ['depth_bins', 'mismatch_plot']
    )['loss_diff']
    .describe()
    .round(0)
)

# AGGREGATE LOSS DISCREPANCIES

print('\n' + '='*80)
print('AGGREGATE LOSS DISCREPANCY BY MISMATCH TYPE')
print('(NSI Loss - Philadelphia Loss) in Millions of Dollars')
print('='*80)

print(
    (
        parcel_gb
        .groupby('mismatch_plot')['loss_diff']
        .sum()
        / 1e6
    )
    .sort_values(ascending=False)
    .round(2)
)

print('\n' + '='*80)
print('TOTAL AGGREGATE LOSS DISCREPANCY')
print('(NSI Loss - Philadelphia Loss) in Millions of Dollars')
print('='*80)

print(
    round(
        parcel_gb['loss_diff'].sum() / 1e6,
        2
    )
)


In [ ]:
# Contextualizing importance of depths

depth_max = parcel_gb[(parcel_gb['mismatch_plot'] != 'NSI Only')]['depth_ft'].max()
anom = parcel_gb[(parcel_gb['mismatch_plot'] == 'NSI Only') & (parcel_gb['depth_ft'] > depth_max)]
dam_anom = anom['loss_diff'].sum()/1e6
count_anom = len(anom)
prop_anom = 1e6*dam_anom/parcel_gb['loss_diff'].sum()

print(f'Anom depth: {depth_max} ft.')
print(f'Total Anom: ${dam_anom}M')
print(f'Count Anom: {count_anom}')
print(f'Prop Anom: {prop_anom}')

In [ ]:
print(nsi_clip_out[nsi_clip_out['fd_id'].isin(anom['comp_id'])][['source', 'ftprntsrc']])

In [ ]:
# NSI ONLY VS LOCATION MATCH COMPARISON

# NSI-only structures

nsi_only_ids = set(
    parcel_gb.loc[
        parcel_gb['mismatch_plot'] == 'NSI Only',
        'comp_id'
    ]
)

nsi_only = (
    nsi_clip_out[
        nsi_clip_out['fd_id'].isin(
            nsi_only_ids
        )
    ][
        ['fd_id', 'source', 'ftprntsrc']
    ]
    .copy()
)

# NSI structures that participate in a location match

analysis_ids = set(
    match_analysis['fd_id']
)

# Remove NSI-only structures

matched_nsi = (
    nsi_clip_out[
        nsi_clip_out['fd_id'].isin(
            analysis_ids - nsi_only_ids
        )
    ][
        ['fd_id', 'source', 'ftprntsrc']
    ]
    .copy()
)

# SOURCE COUNTS

print('\n' + '=' * 80)
print('SOURCE COUNTS')
print('=' * 80)

print('\nNSI ONLY')
print(
    nsi_only['source']
    .value_counts(dropna=False)
)

print('\nLOCATION MATCH')
print(
    matched_nsi['source']
    .value_counts(dropna=False)
)

# SOURCE PROPORTIONS

print('\n' + '=' * 80)
print('SOURCE PROPORTIONS (%)')
print('=' * 80)

print('\nNSI ONLY')
print(
    (
        100 *
        nsi_only['source']
        .value_counts(
            normalize=True,
            dropna=False
        )
    ).round(1)
)

print('\nLOCATION MATCH')
print(
    (
        100 *
        matched_nsi['source']
        .value_counts(
            normalize=True,
            dropna=False
        )
    ).round(1)
)

# FOOTPRINT SOURCE COUNTS

print('\n' + '=' * 80)
print('FTPRNTSRC COUNTS')
print('=' * 80)

print('\nNSI ONLY')
print(
    nsi_only['ftprntsrc']
    .value_counts(dropna=False)
)

print('\nLOCATION MATCH')
print(
    matched_nsi['ftprntsrc']
    .value_counts(dropna=False)
)

# FOOTPRINT SOURCE PROPORTIONS

print('\n' + '=' * 80)
print('FTPRNTSRC PROPORTIONS (%)')
print('=' * 80)

print('\nNSI ONLY')
print(
    (
        100 *
        nsi_only['ftprntsrc']
        .value_counts(
            normalize=True,
            dropna=False
        )
    ).round(1)
)

print('\nLOCATION MATCH')
print(
    (
        100 *
        matched_nsi['ftprntsrc']
        .value_counts(
            normalize=True,
            dropna=False
        )
    ).round(1)
)

# SOURCE × FOOTPRINT SOURCE COUNTS

print('\n' + '=' * 80)
print('SOURCE × FTPRNTSRC COUNTS')
print('=' * 80)

print('\nNSI ONLY')
nsi_only_ct = pd.crosstab(
    nsi_only['source'],
    nsi_only['ftprntsrc'],
    dropna=False
)
print(nsi_only_ct)

print('\nLOCATION MATCH')
matched_ct = pd.crosstab(
    matched_nsi['source'],
    matched_nsi['ftprntsrc'],
    dropna=False
)
print(matched_ct)

# SOURCE × FOOTPRINT SOURCE ROW %

print('\n' + '=' * 80)
print('SOURCE × FTPRNTSRC ROW PERCENTAGES (%)')
print('=' * 80)

print('\nNSI ONLY')

print(
    (
        100 *
        pd.crosstab(
            nsi_only['source'],
            nsi_only['ftprntsrc'],
            normalize='index',
            dropna=False
        )
    ).round(1)
)

print('\nLOCATION MATCH')

print(
    (
        100 *
        pd.crosstab(
            matched_nsi['source'],
            matched_nsi['ftprntsrc'],
            normalize='index',
            dropna=False
        )
    ).round(1)
)

# SOURCE × FOOTPRINT SOURCE OVERALL %

print('\n' + '=' * 80)
print('SOURCE × FTPRNTSRC OVERALL PERCENTAGES (%)')
print('=' * 80)

print('\nNSI ONLY')

print(
    (
        100 *
        pd.crosstab(
            nsi_only['source'],
            nsi_only['ftprntsrc'],
            normalize='all',
            dropna=False
        )
    ).round(1)
)

print('\nLOCATION MATCH')

print(
    (
        100 *
        pd.crosstab(
            matched_nsi['source'],
            matched_nsi['ftprntsrc'],
            normalize='all',
            dropna=False
        )
    ).round(1)
)

In [ ]:
# Loss discrepancy metrics

# FULL DATASET

phil_loss_tot = parcel_gb[f'{dam_col}_phil'].sum() / 1e6
nsi_loss_tot = parcel_gb[f'{dam_col}_nsi'].sum() / 1e6

loss_diff_tot = parcel_gb['loss_diff'].sum() / 1e6

pct_diff = (
    parcel_gb['loss_diff'].sum() /
    parcel_gb[f'{dam_col}_phil'].sum()
)

# Relative loss metrics

rel_loss_rmse = np.sqrt(
    mean_squared_error(
        parcel_gb['rel_loss_phil'].fillna(0),
        parcel_gb['rel_loss_nsi'].fillna(0)
    )
)

rel_loss_mae = mean_absolute_error(
    parcel_gb['rel_loss_phil'].fillna(0),
    parcel_gb['rel_loss_nsi'].fillna(0)
)

rel_loss_medae = np.median(
    np.abs(
        parcel_gb['rel_loss_nsi'].fillna(0) -
        parcel_gb['rel_loss_phil'].fillna(0)
    )
)

rel_loss_bias = (
    parcel_gb['rel_loss_nsi'].fillna(0) -
    parcel_gb['rel_loss_phil'].fillna(0)
).mean()

rel_loss_wmape = (
    np.abs(
        parcel_gb['rel_loss_nsi'].fillna(0) -
        parcel_gb['rel_loss_phil'].fillna(0)
    ).sum()
    /
    parcel_gb['rel_loss_phil'].fillna(0).sum()
)

# Absolute loss metrics

loss_rmse = np.sqrt(
    mean_squared_error(
        parcel_gb[f'{dam_col}_phil'].fillna(0),
        parcel_gb[f'{dam_col}_nsi'].fillna(0)
    )
)

loss_rmsle = np.sqrt(
    mean_squared_error(
        np.log1p(parcel_gb[f'{dam_col}_phil'].fillna(0)),
        np.log1p(parcel_gb[f'{dam_col}_nsi'].fillna(0))
    )
)

loss_wmape = (
    np.abs(
        parcel_gb[f'{dam_col}_nsi'].fillna(0) -
        parcel_gb[f'{dam_col}_phil'].fillna(0)
    ).sum()
    /
    parcel_gb[f'{dam_col}_phil'].fillna(0).sum()
)

# SENSITIVITY ANALYSIS SUBSET
# (for taking out values influence)

sens_sub = parcel_gb[
    (parcel_gb['val_struct_nsi'] > 0) &
    (parcel_gb[f'{dam_col}_nsi'] > 0) &
    (parcel_gb['n_phil'] >= 1) &
    (parcel_gb['rel_loss_phil'].notnull())
].copy()

# SAME SUBSET, ORIGINAL PHILLY VALUES

phil_loss_tot_sens = (
    sens_sub[f'{dam_col}_phil'].sum() / 1e6
)

nsi_loss_tot_sens = (
    sens_sub[f'{dam_col}_nsi'].sum() / 1e6
)

loss_diff_tot_sens = (
    sens_sub['loss_diff'].sum() / 1e6
)

pct_diff_sens = (
    sens_sub['loss_diff'].sum() /
    sens_sub[f'{dam_col}_phil'].sum()
)

# Relative loss metrics

rel_loss_rmse_sens = np.sqrt(
    mean_squared_error(
        sens_sub['rel_loss_phil'].fillna(0),
        sens_sub['rel_loss_nsi'].fillna(0)
    )
)

rel_loss_mae_sens = mean_absolute_error(
    sens_sub['rel_loss_phil'].fillna(0),
    sens_sub['rel_loss_nsi'].fillna(0)
)

rel_loss_medae_sens = np.median(
    np.abs(
        sens_sub['rel_loss_nsi'].fillna(0) -
        sens_sub['rel_loss_phil'].fillna(0)
    )
)

rel_loss_bias_sens = (
    sens_sub['rel_loss_nsi'].fillna(0) -
    sens_sub['rel_loss_phil'].fillna(0)
).mean()

rel_loss_wmape_sens = (
    np.abs(
        sens_sub['rel_loss_nsi'].fillna(0) -
        sens_sub['rel_loss_phil'].fillna(0)
    ).sum()
    /
    sens_sub['rel_loss_phil'].fillna(0).sum()
)

# Absolute loss metrics

loss_rmse_sens = np.sqrt(
    mean_squared_error(
        sens_sub[f'{dam_col}_phil'].fillna(0),
        sens_sub[f'{dam_col}_nsi'].fillna(0)
    )
)

loss_rmsle_sens = np.sqrt(
    mean_squared_error(
        np.log1p(sens_sub[f'{dam_col}_phil'].fillna(0)),
        np.log1p(sens_sub[f'{dam_col}_nsi'].fillna(0))
    )
)

loss_wmape_sens = (
    np.abs(
        sens_sub[f'{dam_col}_nsi'].fillna(0) -
        sens_sub[f'{dam_col}_phil'].fillna(0)
    ).sum()
    /
    sens_sub[f'{dam_col}_phil'].fillna(0).sum()
)

# SAME SUBSET, NSI VALUE SENSITIVITY ANALYSIS

phil_loss_tot_adj = (
    sens_sub[f'{dam_col}_phil_adj'].sum() / 1e6
)

loss_diff_tot_adj = (
    sens_sub['loss_diff_adj'].sum() / 1e6
)

pct_diff_adj = (
    sens_sub['loss_diff_adj'].sum() /
    sens_sub[f'{dam_col}_phil_adj'].sum()
)

loss_rmse_adj = np.sqrt(
    mean_squared_error(
        sens_sub[f'{dam_col}_phil_adj'].fillna(0),
        sens_sub[f'{dam_col}_nsi'].fillna(0)
    )
)

loss_rmsle_adj = np.sqrt(
    mean_squared_error(
        np.log1p(sens_sub[f'{dam_col}_phil_adj'].fillna(0)),
        np.log1p(sens_sub[f'{dam_col}_nsi'].fillna(0))
    )
)

loss_wmape_adj = (
    np.abs(
        sens_sub[f'{dam_col}_nsi'].fillna(0) -
        sens_sub[f'{dam_col}_phil_adj'].fillna(0)
    ).sum()
    /
    sens_sub[f'{dam_col}_phil_adj'].fillna(0).sum()
)

# HYBRID LOSSES
# Use phil_adj where possible, but include
# location mismatches in the discrepancy totals
parcel_gb[f'{dam_col}_phil_hybrid'] = (
    parcel_gb[f'{dam_col}_phil_adj']
)

parcel_gb.loc[
    parcel_gb[f'{dam_col}_phil_hybrid'].isnull(),
    f'{dam_col}_phil_hybrid'
] = parcel_gb[f'{dam_col}_phil']

phil_loss_tot_hybrid = (
    parcel_gb[f'{dam_col}_phil_hybrid'].sum() / 1e6
)

loss_diff_tot_hybrid = (
    (
        parcel_gb[f'{dam_col}_nsi']
        -
        parcel_gb[f'{dam_col}_phil_hybrid']
    ).sum() / 1e6
)

pct_diff_hybrid = (
    (
        parcel_gb[f'{dam_col}_nsi']
        -
        parcel_gb[f'{dam_col}_phil_hybrid']
    ).sum()
    /
    parcel_gb[f'{dam_col}_phil_hybrid'].sum()
)

loss_rmse_hybrid = np.sqrt(
    mean_squared_error(
        parcel_gb[f'{dam_col}_phil_hybrid'].fillna(0),
        parcel_gb[f'{dam_col}_nsi'].fillna(0)
    )
)

loss_rmsle_hybrid = np.sqrt(
    mean_squared_error(
        np.log1p(
            parcel_gb[f'{dam_col}_phil_hybrid'].fillna(0)
        ),
        np.log1p(
            parcel_gb[f'{dam_col}_nsi'].fillna(0)
        )
    )
)

loss_wmape_hybrid = (
    np.abs(
        parcel_gb[f'{dam_col}_nsi'].fillna(0)
        -
        parcel_gb[f'{dam_col}_phil_hybrid'].fillna(0)
    ).sum()
    /
    parcel_gb[f'{dam_col}_phil_hybrid'].fillna(0).sum()
)

# PRINT STATEMENTS

print('\n' + '='*70)
print('FULL DATASET')
print('='*70)

print(f'Parcels analyzed: {len(parcel_gb):,}')

print('\nTOTAL LOSSES ($M)')
print('-'*70)
print(f'Philadelphia: {phil_loss_tot:,.2f}')
print(f'NSI:          {nsi_loss_tot:,.2f}')

print('\nDISCREPANCY')
print('-'*70)
print(f'Total Difference ($M): {loss_diff_tot:,.2f}')
print(f'Percent Difference:    {pct_diff:.1%}')

print('\nRELATIVE LOSS METRICS')
print('-'*70)
print(f'Bias:                 {rel_loss_bias:.4f}')
print(f'MAE:                  {rel_loss_mae:.4f}')
print(f'Median Abs Error:     {rel_loss_medae:.4f}')
print(f'RMSE:                 {rel_loss_rmse:.4f}')
print(f'wMAPE:                {rel_loss_wmape:.1%}')

print('\nABSOLUTE LOSS METRICS')
print('-'*70)
print(f'RMSE ($):             {loss_rmse:,.0f}')
print(f'RMSLE:                {loss_rmsle:.4f}')
print(f'wMAPE:                {loss_wmape:.1%}')

print('\n' + '='*70)
print('SENSITIVITY SUBSET (ORIGINAL PHILLY VALUES)')
print('='*70)

print(f'Parcels analyzed: {len(sens_sub):,}')

print('\nTOTAL LOSSES ($M)')
print('-'*70)
print(f'Philadelphia: {phil_loss_tot_sens:,.2f}')
print(f'NSI:          {nsi_loss_tot_sens:,.2f}')

print('\nDISCREPANCY')
print('-'*70)
print(f'Total Difference ($M): {loss_diff_tot_sens:,.2f}')
print(f'Percent Difference:    {pct_diff_sens:.1%}')

print('\nRELATIVE LOSS METRICS')
print('-'*70)
print(f'Bias:                 {rel_loss_bias_sens:.4f}')
print(f'MAE:                  {rel_loss_mae_sens:.4f}')
print(f'Median Abs Error:     {rel_loss_medae_sens:.4f}')
print(f'RMSE:                 {rel_loss_rmse_sens:.4f}')
print(f'wMAPE:                {rel_loss_wmape_sens:.1%}')

print('\nABSOLUTE LOSS METRICS')
print('-'*70)
print(f'RMSE ($):             {loss_rmse_sens:,.0f}')
print(f'RMSLE:                {loss_rmsle_sens:.4f}')
print(f'wMAPE:                {loss_wmape_sens:.1%}')

print('\n' + '='*70)
print('SENSITIVITY SUBSET (NSI STRUCTURE VALUES)')
print('='*70)

print(f'Parcels analyzed: {len(sens_sub):,}')

print('\nTOTAL LOSSES ($M)')
print('-'*70)
print(f'Adjusted Philadelphia: {phil_loss_tot_adj:,.2f}')
print(f'NSI:                   {nsi_loss_tot_sens:,.2f}')

print('\nDISCREPANCY')
print('-'*70)
print(f'Adjusted Difference ($M): {loss_diff_tot_adj:,.2f}')
print(f'Adjusted Percent Diff:    {pct_diff_adj:.1%}')

print('\nABSOLUTE LOSS METRICS')
print('-'*70)
print(f'RMSE ($):             {loss_rmse_adj:,.0f}')
print(f'RMSLE:                {loss_rmsle_adj:.4f}')
print(f'wMAPE:                {loss_wmape_adj:.1%}')

print('\nCHANGE DUE TO VALUE SUBSTITUTION')
print('-'*70)
print(f'Original Difference (%): {pct_diff_sens:.1%}')
print(f'Adjusted Difference (%): {pct_diff_adj:.1%}')

print('\n' + '='*70)
print('FULL DATASET (NSI VALUE HARMONIZATION)')
print('='*70)

print(f'Parcels analyzed: {len(parcel_gb):,}')

print('\nTOTAL LOSSES ($M)')
print('-'*70)
print(f'Adjusted Philadelphia: {phil_loss_tot_hybrid:,.2f}')
print(f'NSI:                   {nsi_loss_tot:,.2f}')

print('\nDISCREPANCY')
print('-'*70)
print(f'Adjusted Difference ($M): {loss_diff_tot_hybrid:,.2f}')
print(f'Adjusted Percent Diff:    {pct_diff_hybrid:.1%}')

print('\nABSOLUTE LOSS METRICS')
print('-'*70)
print(f'RMSE ($):             {loss_rmse_hybrid:,.0f}')
print(f'RMSLE:                {loss_rmsle_hybrid:.4f}')
print(f'wMAPE:                {loss_wmape_hybrid:.1%}')

In [ ]:
# PHILADELPHIA RES1 SUBSET

res1_parcels = (
    match_analysis[
        match_analysis['bfid'].isin(
            phil_inv_ens[
                phil_inv_ens['occtype'] == 'RES1'
            ].index
        )
    ]['parcel_number']
    .dropna()
    .unique()
)

res1_sub = (
    parcel_gb[
        parcel_gb['comp_id'].isin(
            res1_parcels
        )
    ]
    .copy()
)

# SUMMARY STATISTICS


res1_pct_diff = (
    res1_sub['loss_diff'].sum()
    /
    res1_sub[f'{dam_col}_phil'].sum()
)

res1_rel_wmape = (
    np.abs(
        res1_sub['rel_loss_nsi'].fillna(0)
        -
        res1_sub['rel_loss_phil'].fillna(0)
    ).sum()
    /
    res1_sub['rel_loss_phil'].fillna(0).sum()
)

res1_loss_wmape = (
    np.abs(
        res1_sub[f'{dam_col}_nsi'].fillna(0)
        -
        res1_sub[f'{dam_col}_phil'].fillna(0)
    ).sum()
    /
    res1_sub[f'{dam_col}_phil'].fillna(0).sum()
)

# MISMATCH COMPARISONS

# Flooded parcels (analysis sample)

overall_mismatch = (
    100 *
    parcel_gb['mismatch']
    .value_counts(normalize=True)
).round(2)

res1_mismatch = (
    100 *
    res1_sub['mismatch']
    .value_counts(normalize=True)
).round(2)

# All matched structures

res1_match = (
    matched_df_main[
        matched_df_main['bfid'].isin(
            phil_inv_ens[
                phil_inv_ens['occtype'] == 'RES1'
            ].index
        )
    ]
)

overall_match_mismatch = (
    100 *
    matched_df_main['mismatch']
    .value_counts(normalize=True)
).round(2)

res1_match_mismatch = (
    100 *
    res1_match['mismatch']
    .value_counts(normalize=True)
).round(2)

mismatch_compare = pd.concat(
    [
        overall_mismatch.rename(
            'Flooded Overall (%)'
        ),
        res1_mismatch.rename(
            'Flooded RES1 (%)'
        ),
        overall_match_mismatch.rename(
            'All Structures (%)'
        ),
        res1_match_mismatch.rename(
            'All RES1 (%)'
        )
    ],
    axis=1
).fillna(0)

# PRINT RESULTS

print('\n' + '=' * 70)
print('PHILADELPHIA RES1 SUBSET')
print('=' * 70)

print(
    f'Parcels analyzed: '
    f'{len(res1_sub):,}'
)

print('\nDISCREPANCY')
print('-' * 70)

print(
    f'Percent Difference:  '
    f'{res1_pct_diff:.1%}'
)

print(
    f'Relative Loss wMAPE: '
    f'{res1_rel_wmape:.1%}'
)

print(
    f'Absolute Loss wMAPE: '
    f'{res1_loss_wmape:.1%}'
)

print('\nMISMATCH TYPE COMPARISON')
print('-' * 70)

print(
    mismatch_compare
)

print(
    '\nShare of full flooded sample: '
    f'{len(res1_sub)/len(parcel_gb):.1%}'
)

### Census tract results 

In [ ]:
def create_comparison_dataframe(results_dict,
                                phil_inventory,
                                nsi_inventory, 
                                dam_col,
                                ref_id,
                                result_keys=['phil']):
    """
    Create a dataframe that links building IDs to different reference IDs
    for both Philadelphia and NSI data and aggregates damage and value
    to the level of the reference ID.
    
    Parameters:
    -----------
    results_dict : dict
        Dictionary containing ensemble results
    phil_inventory : DataFrame
        Philadelphia inventory data
    nsi_inventory : DataFrame
        NSI inventory data
    dam_col : str
        Column name for damage values
    ref_id: str
        Name of the spatial reference (e.g., "tract_id"). Must be in
        the phil_refs and nsi_refs dataframes
    result_keys : list, default=['phil']
        Key to access specific results in results_dict
        
    Returns:
    --------
    DataFrame
        Comparison dataframe with damage and property values for both datasets aggregated to
        the level of ref_id
    """
    # Process each result key
    result_dfs = {}
    for key in result_keys:
        # Determine which reference dataframe to use based on key prefix
        if key.startswith('phil'):
            id_col = 'bfid'
            inventory = phil_inventory
        else:
            id_col = 'fd_id'
            inventory = nsi_inventory

        # Process ensemble results
        temp = results_dict[key]
        if ref_id not in temp.columns:
            # temp will have the id as a column, not index
            # but inventory has id as index
            # so reset index on inventory for merge
            temp = temp.merge(inventory.reset_index(), on=id_col)

        # Add dummy sow_ind for the no_unc dataframes
        if 'sow_ind' not in temp.columns:
            temp['sow_ind'] = 1

        temp_gb = temp.groupby(['sow_ind', ref_id]).agg({dam_col: 'sum'}).reset_index()
        loss_by_ref = temp_gb.groupby(ref_id)[dam_col].mean()
        
        ref_vals = inventory.groupby(ref_id).agg({'val_struct': ['median', 'sum', 'size']})
        ref_vals = ref_vals.reset_index()
        ref_vals.columns = [ref_id, 'median_val', 'total_val', 'n_prop']
        
        # Create result dataframe
        result_df = pd.DataFrame({
            dam_col: loss_by_ref,
            'median_val': ref_vals.set_index(ref_id)['median_val'],
            'total_val': ref_vals.set_index(ref_id)['total_val'],
            'n_prop': ref_vals.set_index(ref_id)['n_prop']
        })
        
        result_df = result_df[result_df[dam_col].notnull()]
        result_dfs[key] = result_df

    # Combine all dataframes
    all_dfs = []
    
    # Process result dataframes
    for key, df in result_dfs.items():
        df_reset = df.reset_index()
        df_reset.columns = [ref_id] + [f"{col}_{key.split(':')[0]}" for col in df.columns]
        all_dfs.append(df_reset)

    # Merge all dataframes
    if all_dfs:
        result = all_dfs[0]
        for df in all_dfs[1:]:
            result = result.merge(df, on=ref_id, how='outer')
        
        return result.fillna(0)
    else:
        return pd.DataFrame()

In [ ]:
comp = create_comparison_dataframe(
    all_results, 
    phil_inv_ens[phil_inv_ens.index.isin(match_analysis['bfid'])],
    nsi_inv_ens[nsi_inv_ens.index.isin(match_analysis['fd_id'])],
    dam_col=dam_col,
    ref_id='tract_id',
    result_keys=['no_unc', 'phil', 'nsi_ddfs', 'nsi_unsafe',
                 'nsi_phil', 'nsi_allphil', 'nsiadj_unsafe'],
)

comp_geo = tract_ref.merge(comp, left_on='GEOID', right_on='tract_id')

## Figures

### Figures 1 and 2
Example of inventory matches across datasets for several blocks of Philadelphia residential building footprints and National Structure Inventory (NSI) point locations. The flood depth basemap is from the Hurricane Irene ensemble member with best-fit discharge statistics. The four types of inventory matches are location and characteristics all match (purple outline), location matches but not all characteristics (orange outline), NSI does not represent a Philly structure (red outline), and NSI represents a non-structure (red point not inside a footprint).

We want to make it clear that you need to control for uncertainty in flood vulnerability to isolate the inventory effect on damage discrepancy. Damage discrepancies might look bigger if you don't. This plot aims to show the kinds of mismatches we see, how they vary over space, and how you can get damage agreement/disagreement for either type. This helps justify our large-scale analysis. 

In [ ]:
# Figure 1
# Example tract illustrating inventory agreement
# and disagreement between NSI and Philadelphia


# Create mapping tables from matched_df_main
nsi_map = (
    matched_df_main[
        ['fd_id', 'mismatch']
    ]
    .dropna(subset=['fd_id'])
)

# Drop any property that begins with Philly Only
# since this is out of the NSI sample
nsi_map = nsi_map.loc[
    ~nsi_map['mismatch'].str.startswith(
        'Philly Only'
    )
].copy()

nsi_map['point_class'] = 'Matched NSI Res.'

nsi_map.loc[
    nsi_map['mismatch'].str.startswith(
        'NSI Only',
        na=False
    ),
    'point_class'
] = 'Unmatched NSI Res.'

phil_map = (
    matched_df_main[
        ['bfid', 'mismatch']
    ]
    .dropna(subset=['bfid'])
)

phil_map['poly_class'] = "Characteristics Don't All Match"

phil_map.loc[
    phil_map['mismatch'] == 'DDF Match',
    'poly_class'
] = 'Characteristics All Match'

phil_map.loc[
    phil_map['mismatch'].str.startswith(
        'Philly Only',
        na=False
    ),
    'poly_class'
] = 'Unmatched Philly Res.'

# Drop all polygons that are NSI only
phil_map = phil_map.loc[
    ~phil_map['mismatch'].str.startswith(
        'NSI Only'
    )
].copy()


# Bounding box around tract

t_id = '42101010300'

tract_sub = (
    tract_ref[
        tract_ref['GEOID'] == t_id
    ]
    .to_crs(HAZ_CRS)
)

geom = box(*tract_sub.total_bounds)

clip_box = gpd.GeoDataFrame(
    geometry=[geom],
    crs=HAZ_CRS
)

# Philadelphia polygons (footprints)

phil_geo = (
    phil_inv_out[
        ['bfid', 'geometry']
    ]
    .to_crs(HAZ_CRS)
)

phil_geo = phil_geo.merge(
    phil_map[
        ['bfid', 'poly_class']
    ],
    on='bfid',
    how='inner'
)

phil_temp = gpd.sjoin(
    phil_geo,
    clip_box,
    predicate='intersects'
)

# NSI points

nsi_geo = (
    nsi_clip_out[
        ['fd_id', 'geometry']
    ]
    .to_crs(HAZ_CRS)
)

nsi_geo = nsi_geo.merge(
    nsi_map[
        ['fd_id', 'point_class']
    ],
    on='fd_id',
    how='inner'
)

nsi_temp = gpd.sjoin(
    nsi_geo,
    clip_box,
    predicate='intersects'
)

# Depth grid

dg_filename = HAZ_FILEN.replace(
    '{ens_num}',
    dg_id
)

rift_filep = join(
    HAZ_DIR_UZ,
    dg_filename
)

ens_dg = (
    rio.open_rasterio(
        rift_filep,
        masked=True
    )
    .rio.write_crs(
        HAZ_CRS,
        inplace=True
    )
)

clipped_dg = ens_dg.rio.clip(
    clip_box.geometry.values,
    drop=True,
    invert=False
)

# Plot

fig, ax = plt.subplots(
    figsize=(10, 6),
    dpi=300
)

# Colormap
cmap_with_zero = (
    plt.cm.get_cmap('Blues')
    .copy()
)

cmap_with_zero.set_bad('gray')

zero_mask = (
    clipped_dg <= .01
)

da_masked = clipped_dg[0].where(
    ~zero_mask
)

fg = (
    da_masked * 3.28404
).plot(
    ax=ax,
    cmap=cmap_with_zero,
    alpha=.75,
    vmin=0,
    vmax=8
)

# Philly footprints

phil_temp[
    phil_temp['poly_class']
    == 'Characteristics All Match'
].plot(
    ax=ax,
    edgecolor='purple',
    color='none'
)

phil_temp[
    phil_temp['poly_class']
    == "Characteristics Don't All Match"
].plot(
    ax=ax,
    edgecolor='orange',
    color='none'
)

phil_temp[
    phil_temp['poly_class']
    == 'Unmatched Philly Res.'
].plot(
    ax=ax,
    edgecolor='red',
    color='none'
)

# NSI points

nsi_temp[
    nsi_temp['point_class']
    == 'Matched NSI Res.'
].plot(
    ax=ax,
    color='black',
    markersize=3
)

nsi_temp[
    nsi_temp['point_class']
    == 'Unmatched NSI Res.'
].plot(
    ax=ax,
    color='red',
    markersize=3
)

# Basemap
ax.set_ylim(
    [39.9646, 39.9698]
)

cx.add_basemap(
    ax,
    attribution_size=4,
    source=cx.providers.Esri.WorldImagery,
    crs=phil_temp.crs
)

ax.set_title('')

# Coordinate ticks
xmin, ymin, xmax, ymax = clip_box.total_bounds
xticks = [
    xmin + (xmax - xmin) / 3,
    xmin + 2 * (xmax - xmin) / 3
]

yticks = [
    ymin + (ymax - ymin) / 3,
    ymin + 2 * (ymax - ymin) / 3
]

transformer = Transformer.from_crs(
    HAZ_CRS,
    4326,
    always_xy=True
)

lon_labels = [
    f'{transformer.transform(x, ymin)[0]:.3f}°'
    for x in xticks
]

lat_labels = [
    f'{transformer.transform(xmin, y)[1]:.3f}°'
    for y in yticks
]

ax.set_xticks(
    xticks
)

ax.set_yticks(
    yticks
)

ax.set_xticklabels(
    lon_labels,
    fontsize=10
)

ax.set_yticklabels(
    lat_labels,
    fontsize=10
)

ax.tick_params(
    axis='both',
    direction='out',
    length=4,
    width=1
)

for label in ax.get_yticklabels():

    label.set_rotation(90)

    label.set_ha('center')
    label.set_va('center')

ax.tick_params(
    axis='y',
    pad=10
)

ax.set_xlabel('')
ax.set_ylabel('')

# Colorbar

fig.delaxes(
    fig.axes[-1]
)

cbar_ax = fig.add_axes(
    [0.8, 0.17, 0.02, 0.7]
)

colors_list = (
    ['gray']
    +
    [
        cmap_with_zero(i)
        for i in range(
            cmap_with_zero.N
        )
    ]
)

custom_cmap = colors.ListedColormap(
    colors_list
)

norm = colors.BoundaryNorm(
    [-1, 0, 1, 2, 3, 4, 5, 6, 7, 8],
    custom_cmap.N
)

cbar = ColorbarBase(
    cbar_ax,
    cmap=custom_cmap,
    norm=norm,
    orientation='vertical',
    alpha=0.75
)

cbar.set_ticks(
    np.arange(-.5, 8, 1)
)

cbar.set_ticklabels(
    [
        '0',
        '(0, 1]',
        '(1, 2]',
        '(2, 3]',
        '(3, 4]',
        '(4, 5]',
        '(5, 6]',
        '(6, 7]',
        '>7'
    ]
)

cbar.minorticks_off()

cbar.ax.tick_params(
    labelsize=12
)

cbar.set_label(
    label='Water Depth (Ft.)',
    rotation=270,
    labelpad=20,
    size=14
)

# Legend

legend_elements = [

    Line2D(
        [0],
        [0],
        marker='o',
        ls='',
        color='red',
        label='Unmatched NSI Res.',
        markerfacecolor='red',
        markersize=5
    ),

    Patch(
        facecolor='none',
        edgecolor='red',
        label='Unmatched Philly Res.'
    ),

    Patch(
        facecolor='none',
        edgecolor='orange',
        label="Characteristics Don't All Match"
    ),

    Line2D(
        [0],
        [0],
        marker='o',
        ls='',
        color='black',
        label='Matched NSI Res.',
        markerfacecolor='black',
        markersize=5
    ),

    Patch(
        facecolor='none',
        edgecolor='purple',
        label='Characteristics All Match'
    )

]

ax.legend(
    handles=legend_elements,
    ncol=2,
    loc='center',
    fontsize=12,
    bbox_to_anchor=(.62, -.16)
)

fig.savefig(
    join(FIG_DIR, 'fig1.png'),
    bbox_inches='tight',
    dpi=300
)

In [ ]:
# Finding BFID for plotting purposes

match_examples = (
    parcel_gb[
        parcel_gb['mismatch'] == 'DDF Match'
    ]
)

mismatch_examples = (
    parcel_gb[
        parcel_gb['mismatch'] == 'DDF Mismatch'
    ]
)

common_match = matched_df_main[(matched_df_main['found_type_phil'] == 'B') &
                                (matched_df_main['found_type_nsi'] == 'B') &
                                (matched_df_main['num_story_phil'] == 2) &
                                (matched_df_main['num_story_nsi'] == 2)]

common_misma = matched_df_main[(matched_df_main['found_type_phil'] == 'B') &
                                (matched_df_main['found_type_nsi'] == 'B') &
                                (matched_df_main['num_story_phil'] == 2) &
                                (matched_df_main['num_story_nsi'] == 1)]

# This is a way to find interesting examples for plotting

# mismatch_examples[(mismatch_examples['009_nsi'] == mismatch_examples['009_phil']) &
#                   (mismatch_examples['depth_ft'] < 1) &
#                (mismatch_examples['comp_id'].isin(common_misma['parcel_number'])) &
#                (mismatch_examples['n_nsi'] > 0) &
#                (mismatch_examples['n_phil'] > 0)].sort_values('loss_diff', ascending=False).head(10)

# match_examples[(match_examples['009_nsi'] == match_examples['009_phil']) &
#                   (match_examples['depth_ft'] < 1) &
#                (match_examples['comp_id'].isin(common_match['parcel_number'])) &
#                (match_examples['n_nsi'] > 0) &
#                (match_examples['loss_diff'].abs() < 15000) &
#                (match_examples['n_phil'] > 0)].sort_values('loss_diff', ascending=True).head(10)

# Use the comp_id and search for parcel_number in matched_df_main for the bfid for plotting
# e.g., matched_df_main[matched_df_main['parcel_number'] == '302089410']

In [ ]:
dict_sub = matched_df_main.drop_duplicates('fd_id')
fd_bfid_link = dict(zip(dict_sub['fd_id'], dict_sub['bfid']))
nsi_ddf_plot = nsi_ddf_unc[nsi_ddf_unc['fd_id'].notnull()]
nsi_ddf_plot['bfid'] = nsi_ddf_plot['fd_id'].map(fd_bfid_link)
nsi_ddf_plot[dg_id] = nsi_ddf_plot['fd_id'].map(nsi_depths_df[dg_id])*3.28084
nsi_ddf_plot['rel_loss_nsi'] = nsi_ddf_plot['naccs_loss_009']/nsi_ddf_plot['val_s']

main_phil[dg_id] = main_phil['bfid'].map(phil_depths_df[dg_id])*3.28084

# Load DDFs for plotting
naccs_ddfs = pd.read_parquet(join(VULN_DIR_I, 'physical', 'naccs_ddfs.pqt'))

naccs_2swb = naccs_ddfs[naccs_ddfs['ddf_type'] == '2SWB_RES1']
# Add low/mid/high to naccs_plot
naccs_2swb[['low', 'mid', 'high']] = pd.DataFrame(naccs_2swb['params'].tolist(),
                                                    index=naccs_2swb.index)

naccs_1swb = naccs_ddfs[naccs_ddfs['ddf_type'] == '1SWB_RES1']
# Add low/mid/high to naccs_plot
naccs_1swb[['low', 'mid', 'high']] = pd.DataFrame(naccs_1swb['params'].tolist(),
                                                    index=naccs_1swb.index)

# Create figure with custom grid layout
fig, ax = plt.subplots(figsize=(6, 6), nrows=2, dpi=300,
                       gridspec_kw={'hspace': .3})

# DAMAGE FUNCTION PLOTS

# Define variables and data
bfids = [80084, 28157, 25487]
bfids_m = [50694, 28221, 16646]

for y_name in ['low', 'high']:
    naccs_2swb.plot(x='depth_ft',
                   y=y_name,
                   ax=ax[0],
                   lw=2,
                   ls='--',
                   color='#0077BB')

offsets = {'x': [20, 0, -20],
           'y': [45, -45, 30]}

for i, bfid in enumerate(bfids):
    nsi_bfid_plot = nsi_ddf_plot[nsi_ddf_plot['bfid'] == bfid]
    nsi_bfid_plot['d_adj'] = nsi_bfid_plot[dg_id] - nsi_bfid_plot['ffe']
    nsi_bfid_plot_match = nsi_bfid_plot.groupby('bfid')[['d_adj', 'rel_loss_nsi', 'val_s', dam_col]].mean().reset_index()
    
    phil_bfid_plot = main_phil[main_phil['bfid'] == bfid]
    phil_bfid_plot['d_adj'] = phil_bfid_plot[dg_id] - phil_bfid_plot['ffe']
    phil_bfid_plot_match = phil_bfid_plot.groupby('bfid')[['d_adj', 'rel_loss_phil', 'val_s', dam_col]].mean().reset_index()
    
    sns.scatterplot(data=phil_bfid_plot_match,
                   x='d_adj',
                   y='rel_loss_phil',
                   color='black',
                   ax=ax[0],
                   zorder=10,)
    
    sns.scatterplot(data=nsi_bfid_plot_match,
                   x='d_adj',
                   marker='D',
                   y='rel_loss_nsi',
                   color='black',
                   zorder=10,
                   ax=ax[0],)
    
    anchor_x = float(phil_bfid_plot['d_adj'].iloc[0])
    anchor_y = float(phil_bfid_plot['rel_loss_phil'].iloc[0])
    val_phil = phil_bfid_plot['val_s'].iloc[0]
    val_nsi = nsi_bfid_plot['val_s'].iloc[0]
    discrep = nsi_bfid_plot[dam_col].iloc[0] - phil_bfid_plot[dam_col].iloc[0]

    home_label = f"Home {i + 1}"
    nsi_val_str = f"NSI Value: ${int(round(val_nsi / 1000.0))}K"
    phil_val_str = f"Philly Value: ${int(round(val_phil / 1000.0))}K"
    discrep_str = f"Discrepancy: ${int(round(discrep / 1000.0))}K"
    annotation = f"{home_label}\n{nsi_val_str}\n{phil_val_str}\n{discrep_str}"

    ax[0].annotate(
        annotation,
        xy=(anchor_x, anchor_y),
        xycoords='data',
        xytext=(offsets['x'][i], offsets['y'][i]),
        textcoords='offset points',
        ha='center',
        va='center',
        fontsize=9,
        bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.9, edgecolor="gray"),
    )

for y_name in ['low', 'high']:
    naccs_1swb.plot(x='depth_ft',
                   y=y_name,
                   ax=ax[1],
                   lw=2,
                   ls='dotted',
                   color='firebrick')
    naccs_2swb.plot(x='depth_ft',
                   y=y_name,
                   ax=ax[1],
                   lw=2,
                   ls='--',
                   alpha=.25,
                   color='#0077BB')

offsets = {'x': [-.5, .2, 0],
           'y': [45, -30, -30]}

for i, bfid in enumerate(bfids_m):
    nsi_bfid_plot = nsi_ddf_plot[nsi_ddf_plot['bfid'] == bfid]
    nsi_bfid_plot['d_adj'] = nsi_bfid_plot[dg_id] - nsi_bfid_plot['ffe']
    nsi_bfid_plot_mis = nsi_bfid_plot.groupby('bfid')[['d_adj', 'rel_loss_nsi', 'val_s', dam_col]].mean().reset_index()
    
    phil_bfid_plot = main_phil[main_phil['bfid'] == bfid]
    phil_bfid_plot['d_adj'] = phil_bfid_plot[dg_id] - phil_bfid_plot['ffe']
    phil_bfid_plot_mis = phil_bfid_plot.groupby('bfid')[['d_adj', 'rel_loss_phil', 'val_s', dam_col]].mean().reset_index()
    
    sns.scatterplot(data=phil_bfid_plot_mis,
                   x='d_adj',
                   y='rel_loss_phil',
                   color='black',
                   ax=ax[1],
                   zorder=10,)
    sns.scatterplot(data=nsi_bfid_plot_mis,
                   x='d_adj',
                   y='rel_loss_nsi',
                   marker='D',
                   color='black',
                   zorder=10,
                   ax=ax[1],)
    
    anchor_x = float(phil_bfid_plot_mis['d_adj'].iloc[0])
    anchor_y = float(phil_bfid_plot_mis['rel_loss_phil'].iloc[0])
    val_phil = phil_bfid_plot_mis['val_s'].iloc[0]
    val_nsi = nsi_bfid_plot_mis['val_s'].iloc[0]
    discrep = nsi_bfid_plot_mis[dam_col].iloc[0] - phil_bfid_plot_mis[dam_col].iloc[0]

    home_label = f"Home {i + 4}"
    nsi_val_str = f"NSI Value: ${int(round(val_nsi / 1000.0))}K"
    phil_val_str = f"Philly Value: ${int(round(val_phil / 1000.0))}K"
    discrep_str = f"Discrepancy: ${int(round(discrep / 1000.0))}K"
    annotation = f"{home_label}\n{nsi_val_str}\n{phil_val_str}\n{discrep_str}"


    ax[1].annotate(
        annotation,
        xy=(anchor_x, anchor_y),
        xycoords='data',
        xytext=(offsets['x'][i], offsets['y'][i]),
        textcoords='offset points',
        ha='center',
        va='center',
        fontsize=9,
        bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.9, edgecolor="gray"),
    )

# Configure both damage function plots
for axes, title in zip([ax[0], ax[1]], 
                     ['Depth Damage Functions Match', 
                      'Depth Damage Functions Mismatch']):
    axes.set_ylim([-.025, 1])
    #xlow = naccs_2swb[naccs_2swb['high'] == 0]['depth_ft'].max() - .5
    xhigh = naccs_2swb['depth_ft'].max()
    
    # Remove auto-generated legends
    if axes.get_legend() is not None:
        axes.get_legend().remove()
    
    axes.set_xlim([-5, xhigh])
    axes.set_title(title, size=14)

ax[1].set_xlabel('Depth Relative to First Floor (Ft.)', size=14)
ax[1].tick_params('both', labelsize=12)
ax[0].tick_params('y', labelsize=12)
ax[0].set_xlabel('')
ax[0].tick_params('x', which='both', labelbottom=False)

# Only set y-label on the left plot
ax[0].set_ylabel('Percent Damage', size=14)
ax[1].set_ylabel('Percent Damage', size=14)
ax[0].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, symbol='%', decimals=0))
ax[1].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, symbol='%', decimals=0))

# Remove y-tick labels from right plot
# ax_mismatch.tick_params(axis='y', which='both', labelleft=False)

# Create legend for damage function plots
damage_legend_elements = [
        Line2D([0], [0], marker='o', ls='',
          color='black',
          label='Philly mean percent damage',
          markerfacecolor='black',
          markersize=5),
        Line2D([0], [0], marker='D', ls='',
          label='NSI mean percent damage',
          color='black',
          markerfacecolor='black',
          markersize=5),
        Line2D([0], [0], marker='', ls='--',
          lw=2,
          color='#0077BB',
          alpha=1,
          label='Bounds for two story, w/ basement home'),
        Line2D([0], [0], marker='', ls='dotted',
          lw=2,
          color='firebrick',
          alpha=1,
          label='Bounds for one story, w/ basement home'),
]

# Add annotations
ax[0].annotate('Both Datasets Report Same Location\nAnd 2 Stories w/ Basement',
                 xy=(-4.8, .95),
                 xycoords='data',
                 horizontalalignment='left',
                 verticalalignment='top',
                 size=12)

ax[1].annotate('Both Datasets Report Same Location\nBut NSI Incorrectly Reports 1 Story',
                    xy=(-4.8, .95),
                    xycoords='data',
                    horizontalalignment='left',
                    verticalalignment='top',
                    size=12)

# Add the damage function legend between the two bottom plots
ax[1].legend(handles=damage_legend_elements,
          loc='center',
          ncol=1,
          fontsize=14,
          bbox_to_anchor=(0.45, -.58))

fig.savefig(join(FIG_DIR, 'fig1b.png'), bbox_inches='tight', dpi=300)

### Figure 3
Damage discrepancies across structure inventories by match type and flood depth. Panel A shows the distribution of differences between National Structure Inventory (NSI) and Philadelphia ensembles' mean damage estimates. The red diamonds indicate the mean for each match type and flood depth subgroup. Panel B shows the count of each match type for each flood depth subgroup. Panel C shows the mean cumulative damage discrepancy across the ensemble over depths for each match type and in aggregate. 

In [ ]:
# Keep all mismatch categories
plot_df = parcel_gb.copy()

# ATTRIBUTION DATA

count_share = (
    100 *
    plot_df['mismatch_plot']
    .value_counts(normalize=True)
)

agg_share = (
    plot_df
    .groupby('mismatch_plot')['loss_diff']
    .sum()
)

agg_share = (
    100 *
    agg_share /
    agg_share.sum()
)

attrib_df = pd.DataFrame({
    'Share of Observations':
        count_share,
    'Share of Aggregate Discrepancy':
        agg_share
})

attrib_df.index.name = 'mismatch_plot'

attrib_plot = (
    attrib_df
    .reset_index()
    .melt(
        id_vars='mismatch_plot',
        var_name='Metric',
        value_name='Percent'
    )
)

attrib_plot['Metric'] = pd.Categorical(
    attrib_plot['Metric'],
    categories=[
        'Share of Observations',
        'Share of Aggregate Discrepancy'
    ],
    ordered=True
)

n_obs = len(plot_df)

agg_pct = round(
    100 *
    plot_df['loss_diff'].sum()
    /
    plot_df[f'{dam_col}_phil'].sum()
)

# PLOT SETTINGS

hue_order = [
    'DDF Match',
    'DDF Mismatch',
    'NSI Only',
    'Philly Only',
    'Multiple Buildings\nOn Parcel'
]

palette_dict = {
    'DDF Match': '#0077BB',
    'DDF Mismatch': '#33BBEE',
    'NSI Only': '#CC3311',
    'Philly Only': '#EE7733',
    'Multiple Buildings\nOn Parcel': '#AA4499'
}

# FIGURE LAYOUT

fig = plt.figure(
    figsize=(10, 4),
    dpi=300
)

gs = gridspec.GridSpec(
    2,
    2,
    height_ratios=[2, 1],
    width_ratios=[5, 2],
    wspace=.25,
    hspace=0.1
)

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[1, 0])
ax4 = fig.add_subplot(gs[:, 1])

# RELATIVE LOSS DISCREPANCY

sns.boxplot(
    data=plot_df,
    x='depth_bins',
    y='rel_loss_diff',
    hue='mismatch_plot',
    hue_order=hue_order,
    palette=palette_dict,
    showfliers=False,
    showmeans=True,
    meanprops={
        'markerfacecolor': 'firebrick',
        'markeredgecolor': 'black',
        'marker': 'D'
    },
    ax=ax1
)

ax1.axhline(
    0,
    color='black',
    ls='--',
    alpha=.75
)

ax1.set_xlabel('')
ax1.set_xticklabels([])

ax1.set_ylabel(
    'NSI - Philly\nRelative Loss',
    size=14
)

# COUNTS

sns.countplot(
    data=plot_df,
    x='depth_bins',
    hue='mismatch_plot',
    hue_order=hue_order,
    palette=palette_dict,
    ax=ax2
)

ax2.set_yscale('log')

ax2.set_ylabel(
    'Number of\nObservations',
    size=12
)

ax2.set_xlabel(
    'Depth Relative to Grade (Ft.)',
    size=12
)

# ATTRIBUTION PANEL

sns.barplot(
    data=attrib_plot,
    y='Metric',
    x='Percent',
    hue='mismatch_plot',
    hue_order=hue_order,
    palette=palette_dict,
    ax=ax4
)

ax4.axvline(
    0,
    color='black',
    ls='dotted',
    lw=1
)

ax4.set_title(
    'Share of:',
    size=12
)

ax4.set_xlabel(
    'Percent (%)',
    size=12
)

ax4.set_ylabel('')

ax4.set_xlim(
    [-25, 60]
)

ax4.set_xticks([-40, 0, 50, 100])

ax4.set_yticklabels([
    f'{n_obs:,}\nParcels',
    f'{agg_pct}%\nLoss\nDiscrep.'
])


# LEGEND

for ax in [ax1, ax2, ax4]:

    if ax.get_legend() is not None:
        ax.get_legend().remove()

legend_elements = [
    Patch(
        facecolor=palette_dict[k],
        label=k
    )
    for k in hue_order
]

fig.legend(
    handles=legend_elements,
    fontsize=12,
    loc='upper center',
    bbox_to_anchor=(0.47, -0.01),
    ncol=5
)

# FORMATTING

for ax in [ax1, ax2, ax4]:
    ax.tick_params(
        labelsize=12
    )

fig.align_ylabels(
    [ax1, ax2]
)

fig.tight_layout()

fig.savefig(
    join(FIG_DIR, 'fig3.png'),
    bbox_inches='tight',
    dpi=300
)

In [ ]:
# Should contextualize the mismatch in NSI buildings and Philly buildings for the Multiple Buildings On Parcel
parcel_gb[parcel_gb['mismatch_plot'] == 'Multiple Buildings\nOn Parcel'].groupby(['mismatch_reason'])[['n_nsi', 'n_phil']].sum()

In [ ]:
# Most of these are cases where there's 1 Philadelphia building 
# That means there are a lot of things that might go wrong with 
# depth discrepancies, DDF discrepancies, value discrepancies
parcel_gb[(parcel_gb['mismatch_plot'] == 'Multiple Buildings\nOn Parcel')].groupby(['n_phil', 'n_nsi']).size()

### Figure 4
Damage discrepancies at census tract scale. Panel A shows the wMAPE of discrepancies across tracts. Panel B shows the percent difference between the mean damage of the NSI ensemble accounting for DDF uncertainty only and the mean damage of the Philadelphia ensemble for each census tract. 

In [ ]:
# TRACT-LEVEL WMAPE

comp_tract = (
    match_analysis[
        ['comp_id', 'fd_id', 'bfid']
    ]
    .copy()
)

comp_tract['tract_id'] = (
    comp_tract['bfid']
    .map(
        phil_refs
        .set_index('bfid')['tract_id']
    )
)

comp_tract['tract_id'] = (
    comp_tract['tract_id']
    .fillna(
        comp_tract['fd_id']
        .map(
            nsi_refs
            .set_index('fd_id')['tract_id']
        )
    )
)

comp_tract = (
    comp_tract[
        ['comp_id', 'tract_id']
    ]
    .dropna()
    .drop_duplicates('comp_id')
)

parcel_wmape = (
    parcel_gb
    .merge(
        comp_tract,
        on='comp_id',
        how='left'
    )
)

tract_wmape = (
    parcel_wmape
    .groupby('tract_id')
    .apply(
        lambda x:
        np.abs(
            x[f'{dam_col}_nsi']
            -
            x[f'{dam_col}_phil']
        ).sum()
        /
        x[f'{dam_col}_phil'].sum()
    )
    .rename('wmape')
)

# Merge into comp_geo
comp_geo = comp_geo.merge(
    tract_wmape,
    left_on='tract_id',
    right_index=True,
    how='left'
)

# Do wmape_rel as well
tract_wmape_rel = (
    parcel_wmape
    .groupby('tract_id')
    .apply(
        lambda x:
        np.abs(
            x['rel_loss_nsi']
            -
            x['rel_loss_phil']
        ).sum()
        /
        x['rel_loss_phil'].sum()
        if x['rel_loss_phil'].sum() > 0
        else np.nan
    )
    .rename('wmape_rel')
)

# Merge into comp_geo
comp_geo = comp_geo.merge(
    tract_wmape_rel,
    left_on='tract_id',
    right_index=True,
    how='left'
)

# TRACT-LEVEL RMSE OF RELATIVE LOSS
tract_rmse_rel = (
    parcel_wmape
    .groupby('tract_id')
    .apply(
        lambda x:
        np.sqrt(
            np.mean(
                (
                    x['rel_loss_nsi']
                    -
                    x['rel_loss_phil']
                )**2
            )
        )
    )
    .rename('rmse_rel')
)

# Merge into comp_geo
comp_geo = comp_geo.merge(
    tract_rmse_rel,
    left_on='tract_id',
    right_index=True,
    how='left'
)

In [ ]:
# CREATE FIGURE

fig, ax = plt.subplots(
    figsize=(8, 6),
    ncols=2,
    dpi=300
)

# Keep Web Mercator for basemap support
comp_plot = comp_geo.to_crs(
    epsg=3857
)

# DAMAGE COLUMNS

comp_plot['nsi_loss'] = (
    comp_plot[f'{dam_col}_nsi_ddfs']
    / 1e6
)

comp_plot['phil_loss'] = (
    comp_plot[f'{dam_col}_phil']
    / 1e6
)

# TRACT-LEVEL PERCENT BIAS

comp_plot['dam_bias'] = (
    comp_plot['nsi_loss']
    -
    comp_plot['phil_loss']
)

comp_plot['dam_pct_bias'] = (
    100 *
    comp_plot['dam_bias']
    /
    comp_plot['phil_loss']
)

# TRACT-LEVEL PHYSICAL VULNERABILITY WMAPE

comp_plot['wmape_pct'] = (
    100 *
    comp_plot['wmape_rel']
)

# SUMMARY METRICS

total_nsi_loss = (
    comp_plot['nsi_loss']
    .sum()
)

total_phil_loss = (
    comp_plot['phil_loss']
    .sum()
)

total_pct_dev = np.round(
    100 *
    (
        total_nsi_loss
        -
        total_phil_loss
    )
    /
    total_phil_loss
)

overall_wmape = (
    100 *
    np.abs(
        parcel_gb['rel_loss_nsi']
        -
        parcel_gb['rel_loss_phil']
    ).sum()
    /
    parcel_gb['rel_loss_phil'].sum()
)

# LEFT PANEL
# TRACT-LEVEL PERCENT DISCREPANCY

comp_plot[
    comp_plot['phil_loss'] > 0
].plot(
    ax=ax[0],
    column='dam_pct_bias',
    cmap='bwr',
    legend=True,
    legend_kwds={
        'pad': .03,
        'shrink': .75,
        'extend': 'max'
    },
    norm=colors.TwoSlopeNorm(
        vmin=-50,
        vcenter=0,
        vmax=100
    )
)

# RIGHT PANEL
# TRACT-LEVEL PHYSICAL VULNERABILITY WMAPE

cmap_wmape = plt.cm.get_cmap('YlOrRd')

comp_plot.plot(
    ax=ax[1],
    column='wmape_pct',
    cmap=cmap_wmape,
    legend=True,
    legend_kwds={
        'pad': .03,
        'shrink': .75,
        'extend': 'max'
    },
    vmin=0,
    vmax=50
)

# BASEMAPS

cx.add_basemap(
    ax[0],
    attribution_size=4,
    source=cx.providers.Esri.WorldImagery
)

cx.add_basemap(
    ax[1],
    attribution_size=4,
    source=cx.providers.Esri.WorldImagery
)

# LOCK MAP EXTENT
# Prevent tick generation from expanding the map

xmin, ymin, xmax, ymax = comp_plot.total_bounds

for axes in ax:

    axes.set_xlim(
        xmin,
        xmax
    )

    axes.set_ylim(
        ymin,
        ymax
    )

# CREATE COORDINATE TICKS
xticks = [
    xmin + (xmax - xmin) / 3,
    xmin + 2 * (xmax - xmin) / 3
]

yticks = [
    ymin + (ymax - ymin) / 3,
    ymin + 2 * (ymax - ymin) / 3
]

# CONVERT TO LAT/LON LABELS

transformer = Transformer.from_crs(
    3857,
    4326,
    always_xy=True
)

lon_labels = [
    f'{transformer.transform(x, ymin)[0]:.2f}°'
    for x in xticks
]

lat_labels = [
    f'{transformer.transform(xmin, y)[1]:.2f}°'
    for y in yticks
]

for axes in ax:

    axes.set_xticks(
        xticks
    )

    axes.set_yticks(
        yticks
    )

    axes.set_xticklabels(
        lon_labels,
        fontsize=10
    )

    axes.set_yticklabels(
        lat_labels,
        fontsize=10
    )

    for label in axes.get_yticklabels():

        label.set_rotation(90)
        label.set_ha('center')
        label.set_va('center')
        label.set_rotation_mode('anchor')

    axes.tick_params(
        axis='y',
        pad=10
    )

# TITLES

ax[0].set_title(
    "Discrepancy (%) Introduced by\nNational Structure Inventory",
    fontsize=14,
    pad=12
)

ax[1].set_title(
    "Physical Vulnerability\nWeighted MAPE (%)",
    fontsize=14,
    pad=12
)

# ANNOTATIONS

ax[0].annotate(
    f'Total = {int(total_pct_dev)} %',
    xy=(.04, .94),
    xycoords='axes fraction',
    color='black',
    backgroundcolor='white',
    size=14
)

ax[1].annotate(
    f'Total = {int(round(overall_wmape))} %',
    xy=(.04, .94),
    xycoords='axes fraction',
    color='black',
    backgroundcolor='white',
    size=14
)

# COLORBAR FONT SIZE

for axes in fig.axes:

    if axes._axes.get_label() == '<colorbar>':

        axes.tick_params(
            labelsize=14
        )

fig.savefig(
    join(FIG_DIR, 'fig4.png'),
    bbox_inches='tight',
    dpi=300
)

In [ ]:
len(comp_plot[comp_plot['dam_pct_bias'] >= 100])/len(comp_plot)

In [ ]:
len(comp_plot[comp_plot['dam_pct_bias'] <= 0])/len(comp_plot)

### Figure 5
Plot skill metrics. 

Performance metrics across six procedures with varying locations and treatment of uncertainty. Panel A and B show total damage discrepancy of the procedures in million dollars and percentage, respectively. Panel C shows the root mean square error of damage discrepancy across census tracts. Panel D shows the percentage of census tracts with the correct ranking inside the top 10th percentile of most damaged tracts. 


In [ ]:
def plot_skill_metrics_box(metrics_data, exps, titles, colors, figsize=(12, 8)):
    """
    Create a 2×2 plot of skill metrics using seaborn boxplots with direction indicators.
    
    Parameters:
    -----------
    metrics_data : dict
        Dictionary with experiment names as keys and pandas DataFrames of metrics as values.
    exps : list
        List of experiment names to include
    titles : list
        List of display titles for each experiment
    colors : list
        List of colors for each experiment
    figsize : tuple
        Figure size
    """
    # Define the metrics to plot and their labels, along with preferred direction
    # (True = higher is better, False = lower is better)
    metrics_to_plot = [
        ('total_discrepancy_dollar', 'Total Discrepancy ($ Millions)', False),
        ('total_discrepancy_pct', 'Total Discrepancy (%)', False),
        ('rmse_tract', 'Tract Root Mean Square Error ($ M)', False),
        ('type1_pct', 'Misclassified as Top 10% Damaged Tracts', False),
    ]
    
    # Set up the figure with a 2×2 grid
    fig, axes = plt.subplots(2, 2, figsize=figsize, dpi=300,
                            gridspec_kw={'wspace': 0.2, 'hspace': 0.25})
    axes = axes.flatten()
    
    # Create a color palette dictionary for consistent coloring
    palette_dict = dict(zip(exps, colors))
    
    # Define which boxplots to emphasize
    primary_index = 0    # Most common way people use NSI (leftmost)
    secondary_index = 1  # Main comparison evaluated (second from left)

    # Create each subplot
    for i, (metric_key, metric_label, higher_is_better) in enumerate(metrics_to_plot):
        ax = axes[i]
        
        # Filter data for this metric
        metric_data = metrics_data[metrics_data['metric'] == metric_key]
        
        # Determine the optimal value
        if higher_is_better:
            optimal_value = 100
        else:
            optimal_value = 0
        
        # Add a horizontal line at the optimal value (behind the boxplots)
        ax.axhline(y=optimal_value, color='green', linestyle='--', 
                  alpha=0.7, linewidth=1.5, zorder=1)
        
        # Add background highlights for important boxplots
        y_min, y_max = metric_data['value'].min(), metric_data['value'].max()
        padding = 0.1 * (y_max - y_min)
        y_min -= padding
        y_max += padding
        
        # Add background highlight for primary boxplot
        # ax.axvspan(primary_index - 0.4, primary_index + 0.4, 
        #           color='lightgray', alpha=0.3, zorder=0)
        
        # Add background highlight for secondary boxplot
        ax.axvspan(secondary_index - 0.4, secondary_index + 0.4, 
                  color='lightgray', alpha=0.3, zorder=0)
        
        # Add optimal value line
        ax.axhline(y=optimal_value, color='green', linestyle='--', 
                  alpha=0.7, linewidth=1.5, zorder=1)

        # Create seaborn boxplot (with higher zorder to appear in front)
        sns.boxplot(
            x='experiment',
            y='value',
            data=metric_data,
            palette=palette_dict,
            width=0.7,
            showfliers=True, 
            fill=False,
            ax=ax,
            zorder=2  # Ensure boxplots are drawn on top of the line
        )
        
        # Set title only - no axis labels
        ax.set_title(metric_label, fontsize=14, pad=15)
        ax.set_xlabel('')  # Remove x-label
        ax.set_ylabel('')  # Remove y-label

        ax.tick_params(labelsize=12)
        
        # Format y-axis for percentage metrics
        if 'pct' in metric_key:
            ax.yaxis.set_major_formatter(FuncFormatter(lambda y, _: f'{y:.0f}%'))
        
        # Add grid
        ax.grid(axis='y', linestyle='--', alpha=0.3, linewidth=0.5)
        
        # Remove top and right spines
        sns.despine(ax=ax)
        
        # Remove x-tick labels and ticks
        ax.set_xticklabels([])
        ax.set_xticks([])
    
    # Create legend elements
    legend_elements = []
    
    # Add experiment colors to legend
    for i, title in enumerate(titles):
        legend_elements.append(Patch(facecolor=colors[i], label=title))
    
    # Add optimal value line to legend
    legend_elements.append(Line2D([0], [0], color='green', linestyle='--',
                                 label='Target Value', alpha=0.7, linewidth=1.5))
    
    # Add the legend at the bottom of the figure
    l = fig.legend(handles=legend_elements,
              loc='lower center',
              frameon=True,
              fancybox=False,
              fontsize=12,
              labelspacing=.75,
              ncol=2,  # Two columns for better layout
              bbox_to_anchor=(0.5, -0.05),  # Position below the plots
              edgecolor='black')
    
    for i, text in enumerate(l.get_texts()):
        if i == secondary_index:
            text.set_backgroundcolor("#d3d3d34d")

    # Adjust bottom margin to make room for the legend
    fig.tight_layout()
    fig.subplots_adjust(bottom=0.15)
    
    return fig, axes

In [ ]:
exps = ['nsi_nounc', 'nsi_ddfs', 'nsi_unsafe', 'nsi_phil', 'nsi_allphil', 'nsiadj_unsafe']
titles = ['Status Quo: Standard NSI application',
          'Main Comparison of Study: Sample damage uncertainty',
          'NSI + Enhanced Uncertainty: Sample structure uncertainty',
          'Refinement 1 (R1): Adjust for local characteristics',
          'R2: R1 + Rough structure value calibration',
          'R3: Use local footprints and structure values']

# Define distinct colors for each method
hues = ['#6EA6CD', '#98CAE1', '#C2E4EF', '#FDB366', '#F67E4B', '#DD3D2D']

# Create the plot
metric_ens = pd.read_parquet(join(FO, 'ensembles', 'all_metrics_main.pqt'))
fig, axes = plot_skill_metrics_box(metric_ens, exps, titles, hues)
fig.savefig(join(FIG_DIR, 'fig4.png'), bbox_inches='tight', dpi=300)

In [ ]:
metric_ens[(metric_ens['metric'] == 'total_discrepancy_dollar')].groupby('experiment')['value'].median()

In [ ]:
metric_ens[(metric_ens['experiment'] == 'nsi_ddfs')
           & (metric_ens['metric'] == 'type1_pct')]['value'].describe()

In [ ]:
len(metric_ens[(metric_ens['experiment'] == 'nsi_ddfs') &
               (metric_ens['metric'] == 'type1_pct') &
               (metric_ens['value'] > 27)])

In [ ]:
len(metric_ens[(metric_ens['experiment'] == 'nsi_ddfs') &
               (metric_ens['metric'] == 'type1_pct') &
               (metric_ens['value'] > 36)])

In [ ]:
metric_ens[(metric_ens['experiment'] == 'nsiadj_unsafe')
           & (metric_ens['metric'] == 'type1_pct')]['value'].describe()

In [ ]:
comp_tract = (
    match_analysis[
        ['comp_id', 'fd_id', 'bfid']
    ]
    .copy()
)

comp_tract['tract_id'] = (
    comp_tract['bfid']
    .map(
        phil_refs
        .set_index('bfid')['tract_id']
    )
)

comp_tract['tract_id'] = (
    comp_tract['tract_id']
    .fillna(
        comp_tract['fd_id']
        .map(
            nsi_refs
            .set_index('fd_id')['tract_id']
        )
    )
)

comp_tract = (
    comp_tract[
        ['comp_id', 'tract_id']
    ]
    .dropna()
    .drop_duplicates('comp_id')
)

valadj_check = (
    parcel_gb
    .merge(
        comp_tract,
        on='comp_id',
        how='left'
    )
)

comp_df = valadj_check.groupby('tract_id')[[f'{dam_col}_nsi',
                                            f'{dam_col}_phil',
                                            f'{dam_col}_phil_adj']].sum().reset_index()
baseline_loss = comp_df[f'{dam_col}_phil']
baseline_loss_adj = comp_df[f'{dam_col}_phil_adj']
exp_loss = comp_df[f'{dam_col}_nsi']

top10_threshold = int(len(comp_df) * 0.1)

top10_tracts = baseline_loss.nlargest(top10_threshold).index
top10_tracts_adj = baseline_loss_adj.nlargest(top10_threshold).index
top10_tracts_exp = exp_loss.nlargest(top10_threshold).index

type1_error_adj = len(set(top10_tracts_adj) - set(top10_tracts_exp))
type1_error = len(set(top10_tracts) - set(top10_tracts_exp))

print(f'Number of top 10% tracts: {top10_threshold}')
print(f'Misclassified as top 10% tract, base: {type1_error}')
print(f'Misclassified as top 10% tract, val_adj: {type1_error_adj}')

## Supplementary Figures and Analysis

In [ ]:
FIG_DIR_SUPP = join(FIG_DIR, 'supp')
Path(FIG_DIR_SUPP).mkdir(parents=True, exist_ok=True)

### Maps to provide additional context for processing limitations, mismatches, and loss discrepancies

First, revisiting the `anom` dataframe from postprocessing, there are two clusters of No Parcel Match where NSI properties don't correspond to any structure at all and an illustrative example of NSI getting the wrong number of stories on a 4S building. These will be helpful to illustrate the types of mismatches we're dealing with. 

```{python}
# anom[['comp_id', 'loss_diff', 'rel_loss_diff', 'depth_ft', 'mismatch_reason']].sort_values('loss_diff')
```

FD_ID 572012791, 572012793, 572012794, 572012795 are one group of NSI in the middle of a road. The depths are very high and these contribute a large loss discrepancy. 

FD_ID 572012766, 572012767, and 572012768 are another. It looks like a forested area. 

Parcel number 881122100 is an example of a clear four story building that NSI says is a 1 story building with a basement. This is the largest loss discrepancy of these loss anomalies. The second largest is 881121900, which NSI says is 1 story without a basement. Both are clearly 4 stories from street-level data and the assessment data. 

In [ ]:
### Helper function
def plot_anomaly_map(
    fd_ids=None,
    buffer_dist=.001,
    filename='anomaly_map.png'
):

    # --------------------------------------------------
    # INPUT CHECKS
    # --------------------------------------------------

    if (fd_ids is None):

        raise ValueError(
            'Specify fd_ids'
        )

    # --------------------------------------------------
    # TARGET GEOMETRY
    # --------------------------------------------------

    target_geom = (
        nsi_clip_out[
            nsi_clip_out['fd_id'].isin(fd_ids)
        ]
        .to_crs(HAZ_CRS)
    )

    geom = (
        target_geom
        .union_all()
        .buffer(buffer_dist)
    )

    clip_box = gpd.GeoDataFrame(
        geometry=[geom],
        crs=HAZ_CRS
    )



    # --------------------------------------------------
    # NSI STRUCTURES
    # --------------------------------------------------

    nsi_temp = gpd.sjoin(
        nsi_clip_out[
            ['fd_id', 'geometry']
        ].to_crs(HAZ_CRS),
        clip_box,
        predicate='intersects'
    )

    # --------------------------------------------------
    # DEPTH GRID
    # --------------------------------------------------

    dg_filename = HAZ_FILEN.replace(
        '{ens_num}',
        dg_id
    )

    rift_filep = join(
        HAZ_DIR_UZ,
        dg_filename
    )

    ens_dg = (
        rio.open_rasterio(
            rift_filep,
            masked=True
        )
        .rio.write_crs(
            HAZ_CRS,
            inplace=True
        )
    )

    clipped_dg = ens_dg.rio.clip(
        clip_box.geometry.values,
        drop=True,
        invert=False
    )

    # --------------------------------------------------
    # FIGURE
    # --------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(7, 5),
        dpi=300
    )

    # --------------------------------------------------
    # DEPTHS
    # --------------------------------------------------

    cmap_with_zero = (
        plt.cm.get_cmap('Blues')
        .copy()
    )

    cmap_with_zero.set_bad(
        'none'
    )

    zero_mask = (
        clipped_dg <= .01
    )

    da_masked = (
        clipped_dg[0]
        .where(~zero_mask)
    )

    (
        da_masked * 3.28084
    ).plot(
        ax=ax,
        cmap=cmap_with_zero,
        alpha=.75,
        vmin=0,
        vmax=8
    )

    # --------------------------------------------------
    # NSI STRUCTURES
    # --------------------------------------------------

    if fd_ids is not None:

        nsi_temp[
            nsi_temp['fd_id'].isin(fd_ids)
        ].plot(
            ax=ax,
            color='red',
            markersize=30,
            zorder=10
        )

    # --------------------------------------------------
    # BASEMAP
    # --------------------------------------------------

    cx.add_basemap(
        ax,
        attribution_size=4,
        source=cx.providers.Esri.WorldImagery,
        crs=HAZ_CRS
    )

    # --------------------------------------------------
    # EXTENT
    # --------------------------------------------------

    xmin, ymin, xmax, ymax = (
        clip_box.total_bounds
    )

    ax.set_xlim(
        xmin,
        xmax
    )

    ax.set_ylim(
        ymin,
        ymax
    )

    # --------------------------------------------------
    # COORDINATE TICKS
    # --------------------------------------------------

    xticks = [
        xmin + (xmax - xmin) / 3,
        xmin + 2 * (xmax - xmin) / 3
    ]

    yticks = [
        ymin + (ymax - ymin) / 3,
        ymin + 2 * (ymax - ymin) / 3
    ]

    transformer = Transformer.from_crs(
        HAZ_CRS,
        4326,
        always_xy=True
    )

    lon_labels = [
        f'{transformer.transform(x, ymin)[0]:.4f}°'
        for x in xticks
    ]

    lat_labels = [
        f'{transformer.transform(xmin, y)[1]:.4f}°'
        for y in yticks
    ]

    ax.set_xticks(
        xticks
    )

    ax.set_yticks(
        yticks
    )

    ax.set_xticklabels(
        lon_labels,
        fontsize=10
    )

    ax.set_yticklabels(
        lat_labels,
        fontsize=10
    )

    for label in ax.get_yticklabels():

        label.set_rotation(
            90
        )

        label.set_ha(
            'center'
        )

        label.set_va(
            'center'
        )

    ax.tick_params(
        axis='y',
        pad=8
    )

    ax.set_title('')
    ax.set_xlabel('')
    ax.set_ylabel('')

    # --------------------------------------------------
    # COLORBAR
    # --------------------------------------------------

    fig.delaxes(
        fig.axes[-1]
    )

    cbar_ax = fig.add_axes(
        [0.82, 0.15, 0.025, 0.7]
    )

    custom_cmap = colors.ListedColormap(
        [
            cmap_with_zero(i)
            for i in range(cmap_with_zero.N)
        ]
    )

    norm = colors.BoundaryNorm(
        [0, 1, 2, 3, 4, 5, 6, 7, 8],
        custom_cmap.N
    )

    cbar = ColorbarBase(
        cbar_ax,
        cmap=custom_cmap,
        norm=norm,
        orientation='vertical',
        alpha=0.75
    )

    cbar.set_ticks(
        np.arange(0.5, 8, 1)
    )

    cbar.set_ticklabels(
        [
            '(0,1]',
            '(1,2]',
            '(2,3]',
            '(3,4]',
            '(4,5]',
            '(5,6]',
            '(6,7]',
            '>7'
        ]
    )

    cbar.ax.tick_params(
        labelsize=10
    )

    cbar.set_label(
        'Water Depth (ft)',
        rotation=270,
        labelpad=18,
        size=12
    )

    # --------------------------------------------------
    # LEGEND
    # --------------------------------------------------

    legend_elements = [

        Line2D(
            [0],
            [0],
            marker='o',
            linestyle='',
            color='red',
            markerfacecolor='red',
            markersize=8,
            label='High Consequence NSI Only'
        )

    ]

    ax.legend(
        handles=legend_elements,
        fontsize=10,
        frameon=True,
        loc='upper left'
    )

    fig.savefig(
        join(
            FIG_DIR,
            'supp',
            filename
        ),
        dpi=300,
        bbox_inches='tight'
    )

    return fig, ax

In [ ]:
plot_anomaly_map(
    fd_ids=[
        572012791,
        572012793,
        572012794,
        572012795
    ],
    buffer_dist=.001,
    filename='roadway_false_positive.png'
)

In [ ]:
plot_anomaly_map(
    fd_ids=[
        572012766,
        572012767,
        572012768
    ],
    buffer_dist=.001,
    filename='trees_false_positive.png'
)

Report this example - parcel number 881124500. This is an example of an apartment building with one parcel record and building footprint in the Philly data but 30 NSI records. These 30 NSI records are all over the place. One thinks the structure is more than 3 stories. It is a 3S NB home for structure value purposes and 2SNB home for DDF purposes. Only 2 of the NSI records say 2SNB .12 say 1SWB, 9 say 1SNB, and 7 say 2SWB. Despite this, the NSI structure value is 1.7x as high as the Philly value. The building was built in 1880, which also suggests substantiald depreciation of replacement costs. 

Importantly, the NSI records all say they have parcel source and MBL footprint source. It is unclear how NSI is using the parcel data for this - maybe the core logic data is doing its own additional processing? It's unambiguously one parcel record (processed as apartment, not condo). I could see how the MBL data could treat the rooftop images as different footprints - they do look like multiple attached row homes from satellite imagery and this is likely an example of a building footprint union from our processing. But I see 22 with an eye inspection on the latest Philadelphia imagery (https://property.phila.gov/?p=881124500).

Readers can confirm these numbers with:
```
match_analysis[match_analysis['parcel_number'] == '881124500']['ddf_nsi'].value_counts()
phil_inv_out[phil_inv_out['bfid'] == 78776.0]['year_built']
fdid30 = match_analysis[match_analysis['parcel_number'] == '881124500']['fd_id']
nsi_clip_out[nsi_clip_out['fd_id'].isin(fdid30)][['source', 'ftprntsrc']]
```


### Structure value investigations and sensitivities

In [ ]:
phil_inv_out[phil_inv_out['bld_type'] == 'RES CONDO']

In [ ]:
## CHARACTERIZE VALUE MISMATCHES
# (largely due to occ type mismatches)

# Number of bfids sharing each parcel

parcel_counts = (
    phil_inv_out
    .groupby('parcel_number')
    ['bfid']
    .nunique()
)

phil_inv_out['n_bfid'] = (
    phil_inv_out['parcel_number']
    .map(parcel_counts)
    .fillna(1)
)

# Area used for plotting / value normalization

phil_inv_out['area_plot'] = np.where(
    phil_inv_out['total_livable_area'].notnull(),
    phil_inv_out['total_livable_area'],
    phil_inv_out['total_area']
)

# Divide parcel area across bfids

phil_inv_out['area_plot'] = (
    phil_inv_out['area_plot']
    /
    phil_inv_out['n_bfid']
)

# Value per square foot

phil_inv_out['val_sqft'] = (
    phil_inv_out['val_struct']
    /
    phil_inv_out['area_plot']
)

nsi_clip_out['val_sqft'] = (
    nsi_clip_out['val_struct']
    /
    nsi_clip_out['sqft']
)

nsi_clip_out['area_plot'] = (
    nsi_clip_out['sqft']
)

# LOOKUPS
# Condo sq ft unreliable, exclude from this comparison
phil_inv_sub = phil_inv_out[(phil_inv_out['area_plot'] > 0) &
                            (phil_inv_out['bld_type'] != 'RES CONDO')]
phil_valsqft = dict(
    zip(
        phil_inv_sub['bfid'],
        phil_inv_sub['val_sqft']
    )
)

nsi_valsqft = dict(
    zip(
        nsi_clip_out['fd_id'],
        nsi_clip_out['val_sqft']
    )
)

phil_area = dict(
    zip(
        phil_inv_sub['bfid'],
        phil_inv_sub['area_plot']
    )
)

nsi_area = dict(
    zip(
        nsi_clip_out['fd_id'],
        nsi_clip_out['area_plot']
    )
)

# NSI

nsi_plot = nsi_inv_ens.copy()

nsi_plot['inventory'] = 'NSI'

nsi_plot['ddf'] = (
    nsi_plot['num_story']
    .astype(str)
    + 'S'
    + nsi_plot['found_type']
)

nsi_plot['val_sqft'] = (
    nsi_plot.index
    .map(nsi_valsqft)
)

nsi_plot['area_plot'] = (
    nsi_plot.index
    .map(nsi_area)
)

# PHILLY

phil_plot = phil_inv_ens.copy()

phil_plot['inventory'] = 'Philadelphia'

phil_plot['ddf'] = (
    phil_plot['num_story']
    .astype(str)
    + 'S'
    + phil_plot['found_type']
)

phil_plot['val_sqft'] = (
    phil_plot.index
    .map(phil_valsqft)
)

phil_plot['area_plot'] = (
    phil_plot.index
    .map(phil_area)
)

# COMBINE

val_plot = pd.concat(
    [
        nsi_plot.reset_index(),
        phil_plot.reset_index()
    ],
    ignore_index=True
)

val_plot['occ_group'] = np.where(
    val_plot['occtype'].str.startswith('RES1'),
    'RES1',
    'RES3'
)

val_plot['ddf'] = (
    val_plot['ddf']
    .replace({
        '1SB': '1SWB',
        '1SS': '1SNB',
        '2SB': '2SWB',
        '2SS': '2SNB'
    })
)

val_plot['val_struct_thou'] = (
    val_plot['val_struct']
    / 1000
)

val_plot['area_thou'] = (
    val_plot['area_plot']
    / 1000
)

ddf_order = [
    '1SNB',
    '1SWB',
    '2SNB',
    '2SWB'
]

palette = {
    'NSI': '#77AADD',
    'Philadelphia': '#EE8866'
}

meanprops = {
    'marker': 'D',
    'markerfacecolor': 'black',
    'markeredgecolor': 'black',
    'markersize': 5
}

# FIGURE

fig, axes = plt.subplots(
    4,
    2,
    figsize=(10, 8),
    dpi=300,
    gridspec_kw={
        'height_ratios': [2, 2, 2, 1],
        'wspace': 0.05,
        'hspace': 0.05
    },
    sharex='col',
    sharey='row'
)

# ROW 1
# AREA

for ax, occ in zip(
    axes[0],
    ['RES1', 'RES3']
):

    sns.boxplot(
        data=val_plot[
            (val_plot['occ_group'] == occ) &
            (val_plot['area_plot'] > 0)
        ],
        x='ddf',
        y='area_thou',
        hue='inventory',
        order=ddf_order,
        showfliers=False,
        showmeans=True,
        meanprops=meanprops,
        fill=False,
        linewidth=1.5,
        palette=palette,
        ax=ax
    )

    ax.set_title(
        'Single-Family Dwelling'
        if occ == 'RES1'
        else 'Multi-Family Dwelling',
        fontsize=12
    )

    ax.set_xlabel('')

    ax.set_ylabel(
        'Area\n(1,000 ft²)',
        fontsize=12
    )

    ax.tick_params(
        axis='both',
        labelsize=11
    )

    if ax.get_legend() is not None:
        ax.get_legend().remove()

# ROW 2
# VALUE PER SQ FT

for ax, occ in zip(
    axes[1],
    ['RES1', 'RES3']
):

    sns.boxplot(
        data=val_plot[
            (val_plot['occ_group'] == occ) &
            (val_plot['val_sqft'] > 0)
        ],
        x='ddf',
        y='val_sqft',
        hue='inventory',
        order=ddf_order,
        showfliers=False,
        showmeans=True,
        meanprops=meanprops,
        fill=False,
        linewidth=1.5,
        palette=palette,
        ax=ax
    )

    ax.set_xlabel('')

    ax.set_ylabel(
        'Value per\nSq. Ft. ($)',
        fontsize=12
    )

    ax.tick_params(
        axis='both',
        labelsize=11
    )

    if ax.get_legend() is not None:
        ax.get_legend().remove()

# ROW 3
# STRUCTURE VALUE

for ax, occ in zip(
    axes[2],
    ['RES1', 'RES3']
):

    sns.boxplot(
        data=val_plot[
            val_plot['occ_group'] == occ
        ],
        x='ddf',
        y='val_struct_thou',
        hue='inventory',
        order=ddf_order,
        showfliers=False,
        showmeans=True,
        meanprops=meanprops,
        fill=False,
        linewidth=1.5,
        palette=palette,
        ax=ax
    )

    ax.set_xlabel('')

    ax.set_ylabel(
        'Structure Value\n($1,000s)',
        fontsize=12
    )

    ax.tick_params(
        axis='both',
        labelsize=11
    )

    if ax.get_legend() is not None:
        ax.get_legend().remove()

# ROW 4
# COUNTS

for ax, occ in zip(
    axes[3],
    ['RES1', 'RES3']
):

    sns.countplot(
        data=val_plot[
            val_plot['occ_group'] == occ
        ],
        x='ddf',
        hue='inventory',
        palette=palette,
        order=ddf_order,
        ax=ax
    )

    ax.set_xlabel(
        'Depth Damage Function',
        fontsize=12
    )

    ax.set_ylabel(
        'Count',
        fontsize=12
    )

    ax.tick_params(
        axis='both',
        labelsize=11
    )

    if ax.get_legend() is not None:
        ax.get_legend().remove()

# LEGEND

handles, labels = (
    axes[0, 0]
    .get_legend_handles_labels()
)

handles.append(
    Line2D(
        [0],
        [0],
        marker='D',
        linestyle='',
        color='black',
        markerfacecolor='black',
        label='Mean'
    )
)

labels.append('Mean')

fig.legend(
    handles,
    labels,
    loc='upper center',
    ncol=3,
    fontsize=12,
    frameon=True,
    bbox_to_anchor=(0.5, .03)
)

fig.savefig(
    join(
        FIG_DIR,
        'supp',
        'value_mismatch_occ.png'
    ),
    dpi=300,
    bbox_inches='tight'
)

# STRUCTURE VALUE TABLES

ddf_means = (
    val_plot
    .groupby(
        [
            'occ_group',
            'ddf',
            'inventory'
        ]
    )['val_struct']
    .mean()
    .reset_index()
)

print('\nMEAN STRUCTURE VALUES BY OCCUPANCY AND DDF')
print('-'*60)

print(
    ddf_means.pivot(
        index=['occ_group', 'ddf'],
        columns='inventory',
        values='val_struct'
    ).round(0)
)

# VALUE PER SQ FT TABLES

sqft_medians = (
    val_plot[val_plot['val_sqft'] > 0]
    .groupby(
        [
            'occ_group',
            'ddf',
            'inventory'
        ]
    )['val_sqft']
    .median()
    .reset_index()
)

print('\nMEDIAN VALUE PER SQUARE FOOT BY OCCUPANCY AND DDF')
print('-'*60)

print(
    sqft_medians.pivot(
        index=['occ_group', 'ddf'],
        columns='inventory',
        values='val_sqft'
    ).round(1)
)

# STRUCTURE VALUE RATIOS

ddf_ratio = (
    ddf_means
    .pivot(
        index=['occ_group', 'ddf'],
        columns='inventory',
        values='val_struct'
    )
)

ddf_ratio['Philadelphia / NSI'] = (
    ddf_ratio['Philadelphia']
    /
    ddf_ratio['NSI']
)

ddf_ratio['Literature Analog'] = (
    1
    /
    ddf_ratio['Philadelphia / NSI']
) - 1

lit_table = (
    ddf_ratio[
        [
            'Philadelphia / NSI',
            'Literature Analog'
        ]
    ]
    .copy()
)

lit_table['Philadelphia / NSI'] = (
    lit_table['Philadelphia / NSI']
    .round(3)
)

lit_table['Literature Analog (%)'] = (
    lit_table['Literature Analog']
    * 100
).round(1)

print('\nLITERATURE ANALOG COMPARISON')
print('-'*60)

print(
    lit_table[
        [
            'Philadelphia / NSI',
            'Literature Analog (%)'
        ]
    ]
)

# VALUE DISCREPANCIES
ddf_ratio['Philadelphia / NSI (%)'] = (
    100*(ddf_ratio['Philadelphia'] - ddf_ratio['NSI'])
    /
    ddf_ratio['NSI']
)

print('\nVALUE DISCREPANCY (%)')
print('-'*60)

print(
    ddf_ratio[
        ['Philadelphia / NSI (%)']
    ].round(3)
)

# VALUE PER SQ FT DISCREPANCIES

sqft_ratio = (
    sqft_medians
    .pivot(
        index=['occ_group', 'ddf'],
        columns='inventory',
        values='val_sqft'
    )
)

sqft_ratio['Philadelphia / NSI'] = (
    100*(sqft_ratio['Philadelphia'] - 
    sqft_ratio['NSI'])/sqft_ratio['NSI']
)

print('\nVALUE PER SQ FT DISCREPANCY (%)')
print('-'*60)

print(
    sqft_ratio[
        ['Philadelphia / NSI']
    ].round(3)
)

In [ ]:
# AGGREGATED VALUE COMPARISON
# (collapse across occupancy types)

fig, axes = plt.subplots(
    4,
    1,
    figsize=(7, 10),
    dpi=300,
    gridspec_kw={
        'height_ratios': [2, 2, 2, 1],
        'hspace': 0.05
    },
    sharex=True
)

palette = {
    'NSI': '#77AADD',
    'Philadelphia': '#EE8866'
}

meanprops = {
    'marker': 'D',
    'markerfacecolor': 'black',
    'markeredgecolor': 'black',
    'markersize': 5
}

# AREA

sns.boxplot(
    data=val_plot[
        val_plot['area_plot'] > 0
    ],
    x='ddf',
    y='area_thou',
    hue='inventory',
    order=ddf_order,
    showfliers=False,
    showmeans=True,
    meanprops=meanprops,
    fill=False,
    linewidth=1.5,
    palette=palette,
    ax=axes[0]
)

axes[0].set_ylabel(
    'Building Area\n(1,000 ft²)',
    fontsize=12
)

axes[0].set_xlabel('')

# VALUE PER SQ FT

sns.boxplot(
    data=val_plot[
        val_plot['val_sqft'] > 0
    ],
    x='ddf',
    y='val_sqft',
    hue='inventory',
    order=ddf_order,
    showfliers=False,
    showmeans=True,
    meanprops=meanprops,
    fill=False,
    linewidth=1.5,
    palette=palette,
    ax=axes[1]
)

axes[1].set_ylabel(
    'Value per\nSq. Ft. ($)',
    fontsize=12
)

axes[1].set_xlabel('')

# STRUCTURE VALUE

sns.boxplot(
    data=val_plot,
    x='ddf',
    y='val_struct_thou',
    hue='inventory',
    order=ddf_order,
    showfliers=False,
    showmeans=True,
    meanprops=meanprops,
    fill=False,
    linewidth=1.5,
    palette=palette,
    ax=axes[2]
)

axes[2].set_ylabel(
    'Structure Value\n($1,000s)',
    fontsize=12
)

axes[2].set_xlabel('')

# COUNTS

sns.countplot(
    data=val_plot,
    x='ddf',
    hue='inventory',
    palette=palette,
    order=ddf_order,
    ax=axes[3]
)

axes[3].set_ylabel(
    'Count',
    fontsize=12
)

axes[3].set_xlabel(
    'Depth Damage Function',
    fontsize=12
)

# REMOVE LEGENDS

for ax in axes:

    if ax.get_legend() is not None:
        ax.get_legend().remove()

# SHARED LEGEND

handles, labels = (
    axes[0]
    .get_legend_handles_labels()
)

handles.append(
    Line2D(
        [0],
        [0],
        marker='D',
        linestyle='',
        color='black',
        markerfacecolor='black',
        label='Mean'
    )
)

labels.append('Mean')

fig.legend(
    handles,
    labels,
    loc='upper center',
    ncol=3,
    fontsize=12,
    frameon=True,
    bbox_to_anchor=(0.5, .05)
)

# FORMATTING

for ax in axes:

    ax.tick_params(
        axis='both',
        labelsize=11
    )

fig.align_ylabels(
    axes
)

fig.tight_layout()

fig.savefig(
    join(
        FIG_DIR,
        'supp',
        'value_mismatch.png'
    ),
    dpi=300,
    bbox_inches='tight'
)

# SUMMARY TABLES

overall_area = (
    val_plot[
        val_plot['area_plot'] > 0
    ]
    .groupby(
        ['ddf', 'inventory']
    )['area_plot']
    .median()
    .reset_index()
)

print('\nMEDIAN BUILDING AREA BY DDF')
print('-'*60)

print(
    overall_area.pivot(
        index='ddf',
        columns='inventory',
        values='area_plot'
    ).round(0)
)

overall_area = (
    val_plot[
        val_plot['area_plot'] > 0
    ]
    .groupby(
        ['ddf', 'inventory']
    )['area_plot']
    .mean()
    .reset_index()
)

print('\nMEAN BUILDING AREA BY DDF')
print('-'*60)

print(
    overall_area.pivot(
        index='ddf',
        columns='inventory',
        values='area_plot'
    ).round(0)
)

overall_sqft = (
    val_plot[
        val_plot['val_sqft'] > 0
    ]
    .groupby(
        ['ddf', 'inventory']
    )['val_sqft']
    .median()
    .reset_index()
)

print('\nMEDIAN VALUE PER SQUARE FOOT BY DDF')
print('-'*60)

print(
    overall_sqft.pivot(
        index='ddf',
        columns='inventory',
        values='val_sqft'
    ).round(1)
)


overall_sqft = (
    val_plot[
        val_plot['val_sqft'] > 0
    ]
    .groupby(
        ['ddf', 'inventory']
    )['val_sqft']
    .mean()
    .reset_index()
)

print('\nMEAN VALUE PER SQUARE FOOT BY DDF')
print('-'*60)

print(
    overall_sqft.pivot(
        index='ddf',
        columns='inventory',
        values='val_sqft'
    ).round(1)
)

overall_value = (
    val_plot
    .groupby(
        ['ddf', 'inventory']
    )['val_struct']
    .mean()
    .reset_index()
)

print('\nMEAN STRUCTURE VALUE BY DDF')
print('-'*60)

print(
    overall_value.pivot(
        index='ddf',
        columns='inventory',
        values='val_struct'
    ).round(0)
)

In [ ]:
# MATCHED STRUCTURES ONLY

val_plot_sub = parcel_gb[
    parcel_gb['mismatch'].isin(
        ['DDF Match', 'DDF Mismatch']
    )
].copy()

# LONG FORMAT FOR PLOTTING

# NSI

nsi_plot = val_plot_sub[
    [
        'comp_id',
        'ddf_nsi',
        'sqft_nsi',
        'val_sqft_nsi',
        'val_struct_nsi'
    ]
].copy()

nsi_plot.columns = [
    'comp_id',
    'ddf',
    'sqft',
    'val_sqft',
    'val_struct'
]

nsi_plot['inventory'] = 'NSI'

# PHILLY

phil_plot = val_plot_sub[
    [
        'comp_id',
        'ddf_phil',
        'sqft_phil',
        'val_sqft_phil',
        'val_struct_phil'
    ]
].copy()

phil_plot.columns = [
    'comp_id',
    'ddf',
    'sqft',
    'val_sqft',
    'val_struct'
]

phil_plot['inventory'] = 'Philadelphia'

# Drop RES CONDO buildings, which skew $/sq ft for Philly
condo_parcels = phil_inv_out[phil_inv_out['bld_type'] == 'RES CONDO']['parcel_number']
phil_plot = phil_plot[~phil_plot['comp_id'].isin(condo_parcels)]

val_plot_sub = pd.concat(
    [
        nsi_plot,
        phil_plot[(phil_plot['sqft'] > 0) &
                  (phil_plot['val_struct'] > 0)]
    ],
    ignore_index=True
)

val_plot_sub['sqft_thou'] = (
    val_plot_sub['sqft']
    / 1000
)

val_plot_sub['val_struct_thou'] = (
    val_plot_sub['val_struct']
    / 1000
)

ddf_order = [
    '1SNB',
    '1SWB',
    '2SNB',
    '2SWB'
]

palette = {
    'NSI': '#77AADD',
    'Philadelphia': '#EE8866'
}

meanprops = {
    'marker': 'D',
    'markerfacecolor': 'black',
    'markeredgecolor': 'black',
    'markersize': 5
}

# FIGURE

fig, axes = plt.subplots(
    4,
    1,
    figsize=(7, 10),
    dpi=300,
    gridspec_kw={
        'height_ratios': [2, 2, 2, 1],
        'hspace': 0.05
    },
    sharex=True
)

# AREA

sns.boxplot(
    data=val_plot_sub[
        val_plot_sub['sqft'] > 0
    ],
    x='ddf',
    y='sqft_thou',
    hue='inventory',
    order=ddf_order,
    showfliers=False,
    showmeans=True,
    meanprops=meanprops,
    fill=False,
    linewidth=1.5,
    palette=palette,
    ax=axes[0]
)

axes[0].set_ylabel(
    'Building Area\n(1,000 ft²)',
    fontsize=12
)

axes[0].set_xlabel('')

# VALUE PER SQ FT

sns.boxplot(
    data=val_plot_sub[
        val_plot_sub['val_sqft'] > 0
    ],
    x='ddf',
    y='val_sqft',
    hue='inventory',
    order=ddf_order,
    showfliers=False,
    showmeans=True,
    meanprops=meanprops,
    fill=False,
    linewidth=1.5,
    palette=palette,
    ax=axes[1]
)

axes[1].set_ylabel(
    'Value per\nSq. Ft. ($)',
    fontsize=12
)

axes[1].set_xlabel('')

# STRUCTURE VALUE

sns.boxplot(
    data=val_plot_sub,
    x='ddf',
    y='val_struct_thou',
    hue='inventory',
    order=ddf_order,
    showfliers=False,
    showmeans=True,
    meanprops=meanprops,
    fill=False,
    linewidth=1.5,
    palette=palette,
    ax=axes[2]
)

axes[2].set_ylabel(
    'Structure Value\n($1,000s)',
    fontsize=12
)

axes[2].set_xlabel('')

# COUNTS

sns.countplot(
    data=val_plot_sub,
    x='ddf',
    hue='inventory',
    order=ddf_order,
    palette=palette,
    ax=axes[3]
)

axes[3].set_ylabel(
    'Count',
    fontsize=12
)

axes[3].set_xlabel(
    'Depth Damage Function',
    fontsize=12
)

# REMOVE LEGENDS

for ax in axes:

    if ax.get_legend() is not None:
        ax.get_legend().remove()

# SHARED LEGEND

handles, labels = (
    axes[0]
    .get_legend_handles_labels()
)

handles.append(
    Line2D(
        [0],
        [0],
        marker='D',
        linestyle='',
        color='black',
        markerfacecolor='black',
        label='Mean'
    )
)

labels.append('Mean')

fig.legend(
    handles,
    labels,
    loc='upper center',
    ncol=3,
    fontsize=12,
    frameon=True,
    bbox_to_anchor=(0.5, .05)
)

for ax in axes:

    ax.tick_params(
        axis='both',
        labelsize=11
    )

fig.align_ylabels(
    axes
)

fig.tight_layout()

fig.savefig(
    join(
        FIG_DIR,
        'supp',
        'value_mismatch_matched_structures.png'
    ),
    dpi=300,
    bbox_inches='tight'
)

Using Table 6-2 & Table 6-3 data from Hazus general building stock technical documentation to contextualize both the Philadelphia and NSI $/sq. ft. distributions. 

Most homes in Philly inventory are 100 years old, or older (25th%ile is 100), found using

```
phil_inv_out['bld_yr'] = phil_inv_out['year_built'].astype(float)
(2025 - phil_inv_out[phil_inv_out['bld_yr'] > 0]['bld_yr']).describe()
```

So we can also add a shaded area to denote the depreciation range. 

In [ ]:
from matplotlib.patches import Rectangle, Patch
from matplotlib.lines import Line2D

order = ['1SNB', '1SWB', '2SNB', '2SWB']

res3_ref = [134.57, 118.30, 250.19, 232.69, 217.05, 198.83]

hazus_res1 = {
    '1SNB': [114.63, 128.20, 172.73, 209.44],
    '1SWB': [148.48, 168.75, 234.63, 274.34,
             125.03, 140.95, 196.98, 234.29],
    '2SNB': [121.00, 132.88, 174.96, 212.86,
             121.00, 137.43, 179.34, 218.28],
    '2SWB': [139.95, 158.08, 210.21, 250.56,
             127.75, 140.98, 189.21, 227.86,
             139.95, 156.88, 204.94, 245.93,
             127.75, 143.73, 189.84, 229.53],
}

hazus_env = pd.DataFrame({
    'ddf': order,
    'hazus_min': [
        min(hazus_res1[d] + res3_ref)
        for d in order
    ],
    'hazus_max': [
        max(hazus_res1[d] + res3_ref)
        for d in order
    ],
})

fig, ax = plt.subplots(
    figsize=(10, 6),
    dpi=300
)

# HAZUS envelope first (behind boxplots)

# HAZUS ENVELOPES

for i, row in hazus_env.iterrows():

    depr_min = 0.2 * row['hazus_min']

    # Depreciated-to-minimum range

    ax.add_patch(
        Rectangle(
            (i - 0.4, depr_min),
            0.8,
            row['hazus_min'] - depr_min,
            facecolor='#F0D9A7',
            edgecolor='none',
            alpha=0.6,
            zorder=0
        )
    )

    # HAZUS replacement-cost range

    ax.add_patch(
        Rectangle(
            (i - 0.4, row['hazus_min']),
            0.8,
            row['hazus_max'] - row['hazus_min'],
            facecolor='lightgray',
            edgecolor='none',
            alpha=0.35,
            zorder=1
        )
    )

# Boxplots

sns.boxplot(
    data=val_plot_sub,
    x='ddf',
    y='val_sqft',
    hue='inventory',
    order=ddf_order,
    showfliers=False,
    showmeans=True,
    meanprops={
        'marker': 'D',
        'markerfacecolor': 'black',
        'markeredgecolor': 'black',
        'markersize': 5
    },
    fill=False,
    linewidth=1.5,
    palette=palette,
    ax=ax
)

# Labels

ax.set_ylabel(
    'Value per Sq. Ft. ($)',
    fontsize=12
)

ax.set_xlabel(
    'Depth Damage Function',
    fontsize=12
)

ax.tick_params(
    axis='both',
    labelsize=12
)

# LEGEND

handles, labels = ax.get_legend_handles_labels()

legend_elements = handles.copy()


legend_elements.append(
    Line2D(
        [0],
        [0],
        marker='D',
        linestyle='',
        color='black',
        markerfacecolor='black',
        label='Mean'
    )
)

legend_elements.append(
    Patch(
        facecolor='lightgray',
        edgecolor='none',
        alpha=0.35,
        label='HAZUS replacement cost range (pre-depreciation)'
    )
)

legend_elements.append(
    Patch(
        facecolor='#F0D9A7',
        edgecolor='none',
        alpha=0.6,
        label='Illustrative 80% depreciation adjustment'
    )
)

ax.legend(
    handles=legend_elements,
    loc='upper center',
    bbox_to_anchor=(0.4, .99),
    ncol=2,
    fontsize=12,
    frameon=True
)

fig.tight_layout()

fig.savefig(
    join(
        FIG_DIR,
        'supp',
        'matched_value_sqft_hazus_envelope.png'
    ),
    dpi=300,
    bbox_inches='tight'
)

To help contextualize these distributions, consider Taghinezhad, A., Friedland, C. J., Rohli, R. V., Marx, B. D., Giering, J., & Nahmens, I. (2021). Predictive statistical cost estimation model for existing single family home elevation projects. Frontiers in Built Environment, 7, 646668.

The authors evaluated 139 home elevation projects and found the following summary statistics:

1. Elevation cost - mean = $241,160, median = $179,567 and standard deviation = $172,665

2. Average floor area - mean = 1820, median = 1720, standard deviation = 590 $ft^2$

3. Cost / unit area / unit elevation ($/$ft^2$/ft) - mean = 24, median = 23, standard deviation = 12

The NSI $/sq ft estimates have similar variation as the Philly ones for one story buildings, but less variation for multi story. The coefficient of variation of .5 on unit cost and .71 on elevation cost from the elevation study shows substantial variation around the mean. 

It will help to compare coefficients of variation across inventories and DDF types. 

In [ ]:
def cv(x):

    return np.std(x, ddof=1) / np.mean(x)

cv_table = (
    val_plot_sub
    .groupby(
        ['inventory']
    )
    .agg(
        val_sqft_mean=('val_sqft', 'mean'),
        val_sqft_sd=('val_sqft', 'std'),
        value_mean=('val_struct', 'mean'),
        value_sd=('val_struct', 'std')
    )
)

cv_table['val_sqft_cv'] = (
    cv_table['val_sqft_sd']
    /
    cv_table['val_sqft_mean']
)

cv_table['value_cv'] = (
    cv_table['value_sd']
    /
    cv_table['value_mean']
)

print(
    cv_table[
        [
            'val_sqft_cv',
            'value_cv'
        ]
    ].round(2)
)

The coefficient of variation is higher for the Philadelphia inventory. With the elevation study just focusing on one construction component, I'm inclined to think the Philadelphia inventory results here are more realistic of variation in replacement costs. 

In [ ]:
val_plot_sub = parcel_gb[
    parcel_gb['mismatch'].isin(
        ['DDF Match', 'DDF Mismatch']
    )
].copy()

val_plot_sub = val_plot_sub[val_plot_sub['val_struct_phil'] > 0]

val_plot_sub['value_diff_thou'] = (
    val_plot_sub['value_diff']
    / 1000
)

fig, axes = plt.subplots(
    3,
    1,
    figsize=(7, 8),
    dpi=300,
    sharey=True
)

# AREA DIFFERENCE

sns.ecdfplot(
    data=val_plot_sub,
    x='sqft_diff',
    color='#77AADD',
    linewidth=2,
    ax=axes[0]
)

axes[0].axvline(
    0,
    color='black',
    ls='--'
)

axes[0].set_xlabel(
    'Building Area Difference (ft²)\nNSI − Philadelphia'
)

axes[0].set_ylabel(
    'Cumulative Share'
)

# VALUE PER SQ FT DIFFERENCE

sns.ecdfplot(
    data=val_plot_sub[
        np.isfinite(
            val_plot_sub['val_sqft_diff']
        )
    ],
    x='val_sqft_diff',
    color='#EE8866',
    linewidth=2,
    ax=axes[1]
)

axes[1].axvline(
    0,
    color='black',
    ls='--'
)

axes[1].set_xlabel(
    'Value per Sq. Ft. Difference ($)\nNSI − Philadelphia'
)

axes[1].set_ylabel(
    'Cumulative Share'
)

# STRUCTURE VALUE DIFFERENCE

sns.ecdfplot(
    data=val_plot_sub,
    x='value_diff_thou',
    color='#44AA99',
    linewidth=2,
    ax=axes[2]
)

axes[2].axvline(
    0,
    color='black',
    ls='--'
)

axes[2].set_xlabel(
    'Structure Value Difference ($1,000s)\nNSI − Philadelphia'
)

axes[2].set_ylabel(
    'Cumulative Share'
)


# FORMATTING

for ax in axes:

    ax.tick_params(
        labelsize=11
    )

    ax.set_ylim(
        0,
        1
    )

for ax, var in zip(
    axes,
    [
        'sqft_diff',
        'val_sqft_diff',
        'value_diff_thou'
    ]
):

    p_low, p_high = np.percentile(
        val_plot_sub[var].dropna(),
        [2.5, 97.5]
    )

    ax.set_xlim(
        p_low,
        p_high
    )

    med = (
        val_plot_sub[var]
        .median()
    )

    xmin, xmax = ax.get_xlim()

    frac = (
        med - xmin
    ) / (
        xmax - xmin
    )

    ax.axhline(
        0.5,
        xmin=0,
        xmax=frac,
        color='gray',
        ls=':',
        lw=1.5
    )

    ax.vlines(
        med,
        ymin=0,
        ymax=0.5,
        color='gray',
        ls=':',
        lw=1.5
    )

    ax.scatter(
        med,
        0.5,
        color='black',
        s=25,
        zorder=10
    )

fig.tight_layout()

fig.savefig(
    join(
        FIG_DIR,
        'supp',
        'value_discreps_matched_structures.png'
    ),
    dpi=300,
    bbox_inches='tight'
)

for var in [
    'sqft_diff',
    'val_sqft_diff',
    'value_diff_thou'
]:

    print('\n' + var)

    print(
        val_plot_sub[var]
        .quantile(
            [0.05, 0.25, 0.5, 0.75, 0.95]
        )
    )

In [ ]:
plot_sub = val_plot_sub[
    val_plot_sub['ddf_phil'] == '2SWB'
].copy()

summary_df = (
    plot_sub
    .groupby('ddf_nsi')
    .agg(
        sqft_mean=('sqft_diff', 'mean'),
        sqft_sd=('sqft_diff', 'std'),

        val_sqft_mean=('val_sqft_diff', 'mean'),
        val_sqft_sd=('val_sqft_diff', 'std'),

        value_mean=('value_diff_thou', 'mean'),
        value_sd=('value_diff_thou', 'std'),

        n=('comp_id', 'size')
    )
    .reset_index()
)

order = [
    '2SWB',
    '1SWB',
    '1SNB',
    '2SNB'
]

fig, axes = plt.subplots(
    4,
    1,
    figsize=(6, 8),
    dpi=300,
    sharex=True,
    gridspec_kw={
        'height_ratios': [2, 2, 2, 1],
        'hspace': 0.08
    }
)

sns.pointplot(
    data=plot_sub,
    x='ddf_nsi',
    y='sqft_diff',
    order=order,
    errorbar='sd',
    linestyle='none',
    capsize=.2,
    ax=axes[0]
)

axes[0].axhline(
    0,
    color='lightgray',
    zorder=0,
    lw=2
)

axes[0].set_ylabel(
    'Area Diff.\n(ft²)',
    fontsize=12
)

sns.pointplot(
    data=plot_sub,
    x='ddf_nsi',
    y='val_sqft_diff',
    order=order,
    errorbar='sd',
    linestyle='none',
    capsize=.2,
    ax=axes[1]
)

axes[1].axhline(
    0,
    color='lightgray',
    zorder=0,
    lw=2
)

axes[1].set_ylabel(
    'Value/Sq.Ft.\nDiff. ($)',
    fontsize=12
)

sns.pointplot(
    data=plot_sub,
    x='ddf_nsi',
    y='value_diff_thou',
    order=order,
    errorbar='sd',
    linestyle='none',
    capsize=.2,
    ax=axes[2]
)

axes[2].axhline(
    0,
    color='lightgray',
    zorder=0,
    lw=2
)

axes[2].set_ylabel(
    'Value Diff.\n($1,000s)',
    fontsize=12
)

sns.countplot(
    data=plot_sub,
    x='ddf_nsi',
    order=order,
    color='lightgray',
    edgecolor='black',
    ax=axes[3]
)

axes[3].set_ylabel(
    'Count',
    fontsize=12
)

axes[3].set_xlabel(
    'NSI Depth Damage Function\n(Philadelphia = 2SWB)',
    fontsize=12
)

axes[1].set_yscale('symlog')
axes[2].set_yscale('symlog')

formatter = FuncFormatter(
    lambda x, pos: f'{int(x):,}'
)

axes[1].yaxis.set_major_formatter(
    formatter
)

axes[2].yaxis.set_major_formatter(
    formatter
)

fig.savefig(join(FIG_DIR_SUPP, '2swb_val_discrep.png'),
            bbox_inches='tight', dpi=300)

In [ ]:
plot_sub = val_plot_sub[
    val_plot_sub['ddf_phil'] == '2SNB'
].copy()

summary_df = (
    plot_sub
    .groupby('ddf_nsi')
    .agg(
        sqft_mean=('sqft_diff', 'mean'),
        sqft_sd=('sqft_diff', 'std'),

        val_sqft_mean=('val_sqft_diff', 'mean'),
        val_sqft_sd=('val_sqft_diff', 'std'),

        value_mean=('value_diff_thou', 'mean'),
        value_sd=('value_diff_thou', 'std'),

        n=('comp_id', 'size')
    )
    .reset_index()
)

order = [
    '2SNB',
    '1SWB',
    '1SNB',
    '2SWB'
]

fig, axes = plt.subplots(
    4,
    1,
    figsize=(6, 8),
    dpi=300,
    sharex=True,
    gridspec_kw={
        'height_ratios': [2, 2, 2, 1],
        'hspace': 0.08
    }
)

sns.pointplot(
    data=plot_sub,
    x='ddf_nsi',
    y='sqft_diff',
    order=order,
    errorbar='sd',
    linestyle='none',
    capsize=.2,
    ax=axes[0]
)

axes[0].axhline(
    0,
    color='lightgray',
    zorder=0,
    lw=2
)

axes[0].set_ylabel(
    'Area Diff.\n(ft²)',
    fontsize=12
)

sns.pointplot(
    data=plot_sub,
    x='ddf_nsi',
    y='val_sqft_diff',
    order=order,
    errorbar='sd',
    linestyle='none',
    capsize=.2,
    ax=axes[1]
)

axes[1].axhline(
    0,
    color='lightgray',
    zorder=0,
    lw=2
)

axes[1].set_ylabel(
    'Value/Sq.Ft.\nDiff. ($)',
    fontsize=12
)

sns.pointplot(
    data=plot_sub,
    x='ddf_nsi',
    y='value_diff_thou',
    order=order,
    errorbar='sd',
    linestyle='none',
    capsize=.2,
    ax=axes[2]
)

axes[2].axhline(
    0,
    color='lightgray',
    zorder=0,
    lw=2
)

axes[2].set_ylabel(
    'Value Diff.\n($1,000s)',
    fontsize=12
)

sns.countplot(
    data=plot_sub,
    x='ddf_nsi',
    order=order,
    color='lightgray',
    edgecolor='black',
    ax=axes[3]
)

axes[3].set_ylabel(
    'Count',
    fontsize=12
)

axes[3].set_xlabel(
    'NSI Depth Damage Function\n(Philadelphia = 2SNB)',
    fontsize=12
)

axes[0].set_yscale('symlog')
axes[1].set_yscale('symlog')
axes[2].set_yscale('symlog')

formatter = FuncFormatter(
    lambda x, pos: f'{int(x):,}'
)

axes[0].yaxis.set_major_formatter(
    formatter
)

axes[1].yaxis.set_major_formatter(
    formatter
)

axes[2].yaxis.set_major_formatter(
    formatter
)

fig.savefig(join(FIG_DIR_SUPP, '2snb_val_discrep.png'),
            bbox_inches='tight', dpi=300)

### Breaking down the mismatch types in more detail

In [ ]:
# NSI-ONLY LOCATION ERRORS


nsi_only = (
    parcel_gb[
        parcel_gb['mismatch_plot'] == 'NSI Only'
    ]
    .copy()
)

nsi_only_summary = (
    nsi_only
    .groupby('mismatch_reason')
    .agg(
        count=('loss_diff', 'size'),
        total_discrepancy=('loss_diff', 'sum'),
        mean_discrepancy=('loss_diff', 'mean')
    )
)

nsi_only_summary['share_obs (%)'] = (
    100 *
    nsi_only_summary['count']
    /
    nsi_only_summary['count'].sum()
)

nsi_only_summary['share_discrepancy (%)'] = (
    100 *
    nsi_only_summary['total_discrepancy']
    /
    nsi_only_summary['total_discrepancy'].sum()
)

nsi_only_summary['total_discrepancy ($M)'] = (
    nsi_only_summary['total_discrepancy']
    / 1e6
)

nsi_only_summary['mean_discrepancy ($)'] = (
    nsi_only_summary['mean_discrepancy']
)


# PRINT RESULTS

print('\n' + '=' * 80)
print('NSI-ONLY LOCATION ERRORS')
print('=' * 80)

print(
    nsi_only_summary[
        [
            'count',
            'share_obs (%)',
            'total_discrepancy ($M)',
            'share_discrepancy (%)',
            'mean_discrepancy ($)'
        ]
    ]
    .round(2)
)


In [ ]:
# SUBSET TO LOCATION-MATCHED RECORDS

plot_df = (
    parcel_gb[
        parcel_gb['mismatch_plot'].isin(
            [
                'DDF Match',
                'DDF Mismatch'
            ]
        )
    ]
    .copy()
)

plot_df['match_type'] = np.where(
    plot_df['009_phil']
    == plot_df['009_nsi'],
    'Depth Match\n',
    'Depth Mismatch\n'
)

plot_df['match_type'] = (
    plot_df['match_type']
    +
    plot_df['mismatch_plot']
)

plot_df['loss_diff_mil'] = (
    plot_df['loss_diff']
    / 1e6
)

hue_order = [
    'Depth Match\nDDF Match',
    'Depth Match\nDDF Mismatch',
    'Depth Mismatch\nDDF Match',
    'Depth Mismatch\nDDF Mismatch'
]

palette_dict = {
    'Depth Match\nDDF Match':
        '#0077BB',
    'Depth Match\nDDF Mismatch':
        '#66CCEE',

    'Depth Mismatch\nDDF Match':
        '#CC3311',
    'Depth Mismatch\nDDF Mismatch':
        '#EE7733'
}

# --------------------------------------------------
# FIGURE
# --------------------------------------------------

fig, (ax1, ax2, ax3) = plt.subplots(
    3,
    1,
    figsize=(8, 7),
    dpi=300,
    sharex=True,
    gridspec_kw={
        'height_ratios': [2, 1, 1],
        'hspace': 0.1
    }
)

# --------------------------------------------------
# RELATIVE LOSS DISCREPANCY
# --------------------------------------------------

sns.boxplot(
    data=plot_df,
    x='depth_bins',
    y='rel_loss_diff',
    hue='match_type',
    hue_order=hue_order,
    palette=palette_dict,
    showfliers=False,
    showmeans=True,
    meanprops={
        'markerfacecolor': 'firebrick',
        'markeredgecolor': 'black',
        'marker': 'D'
    },
    ax=ax1
)

ax1.axhline(
    0,
    color='black',
    ls='--',
    alpha=.75
)

ax1.set_ylabel(
    'NSI - Philly\nRelative Loss',
    size=14
)

ax1.set_xlabel('')

# --------------------------------------------------
# COUNTS
# --------------------------------------------------

sns.countplot(
    data=plot_df,
    x='depth_bins',
    hue='match_type',
    hue_order=hue_order,
    palette=palette_dict,
    ax=ax2
)

ax2.set_yscale('log')

ax2.set_ylabel(
    'Number of\nObservations',
    size=12
)

ax2.set_xlabel('')

# --------------------------------------------------
# AGGREGATE LOSS DISCREPANCY
# --------------------------------------------------

sns.barplot(
    data=plot_df,
    x='depth_bins',
    y='loss_diff_mil',
    hue='match_type',
    hue_order=hue_order,
    palette=palette_dict,
    estimator='sum',
    errorbar=None,
    ax=ax3
)

ax3.axhline(
    0,
    color='black',
    ls='--',
    alpha=.75
)

ax3.set_ylabel(
    'Aggregate Loss\nDiscrepancy ($M)',
    size=12
)

ax3.set_xlabel(
    'Depth Relative to Grade (Ft.)',
    size=12
)

ax3.set_ylim([-20, 10])
ax3.set_yscale('symlog')

# --------------------------------------------------
# REMOVE SUBPLOT LEGENDS
# --------------------------------------------------

for ax in [ax1, ax2, ax3]:

    if ax.get_legend() is not None:

        ax.get_legend().remove()

# --------------------------------------------------
# SHARED LEGEND
# --------------------------------------------------

legend_elements = [

    Line2D(
        [],
        [],
        linestyle='none',
        label='Depth Match'
    ),

    Patch(
        facecolor='#0077BB',
        label='DDF Match'
    ),

    Patch(
        facecolor='#66CCEE',
        label='DDF Mismatch'
    ),

    Line2D(
        [],
        [],
        linestyle='none',
        label='Depth Mismatch'
    ),

    Patch(
        facecolor='#CC3311',
        label='DDF Match'
    ),

    Patch(
        facecolor='#EE7733',
        label='DDF Mismatch'
    )

]

fig.legend(
    handles=legend_elements,
    fontsize=11,
    loc='upper center',
    bbox_to_anchor=(0.5, .02),
    ncol=2
)

# --------------------------------------------------
# FORMATTING
# --------------------------------------------------

for ax in [ax1, ax2, ax3]:

    ax.tick_params(
        labelsize=12
    )

fig.align_ylabels(
    [ax1, ax2, ax3]
)

fig.tight_layout()

fig.savefig(
    join(
        FIG_DIR,
        'supp',
        'depth_vs_ddf_discrepancy.png'
    ),
    dpi=300,
    bbox_inches='tight'
)

In [ ]:
plot_sub = plot_df[
    plot_df['009_nsi']
    !=
    plot_df['009_phil']
].copy()

plot_sub['depth_diff'] = (
    plot_sub['009_nsi'].fillna(0)
    -
    plot_sub['009_phil'].fillna(0)
)

plot_sub['nsi_story'] = np.where(
    plot_sub['ddf_nsi'].str.startswith('1'),
    '1 Story',
    '2 Story'
)

plot_sub['nsi_found'] = np.where(
    plot_sub['ddf_nsi'].str.endswith('WB'),
    'Basement',
    'No Basement'
)

phil_order = [
    '2SWB',
    '2SNB'
]

nsi_order = [
    '2SWB',
    '1SWB',
    '1SNB',
    '2SNB'
]

palette = {
    'Basement': '#0077BB',
    'No Basement': '#EE7733'
}

story_order = [
    '1 Story',
    '2 Story'
]

fig, axes = plt.subplots(
    2,
    2,
    figsize=(8, 6),
    dpi=300,
    sharex=True,
    sharey=True
)

for i, phil_ddf in enumerate(phil_order):

    for j, story in enumerate(story_order):

        ax = axes[i, j]

        temp = (
            plot_sub[
                (plot_sub['ddf_phil'] == phil_ddf)
                &
                (plot_sub['nsi_story'] == story)
            ]
        )

        sns.scatterplot(
            data=temp,
            x='depth_diff',
            y='rel_loss_diff',
            hue='nsi_found',
            hue_order=[
                'Basement',
                'No Basement'
            ],
            palette=palette,
            alpha=.6,
            s=35,
            ax=ax
        )

        ax.axhline(
            0,
            color='black',
            ls='--',
            lw=1,
            alpha=.75
        )

        ax.axvline(
            0,
            color='black',
            ls='--',
            lw=1,
            alpha=.75
        )


        # Column titles

        if i == 0:

            ax.set_title(
                f'NSI = {story}',
                fontsize=12
            )

        # Row labels

        if j == 0:

            ax.set_ylabel(
                'Relative Loss Difference\n('
                f'Philadelphia = {phil_ddf})',
                fontsize=12
            )

        else:

            ax.set_ylabel('')

        ax.tick_params(
            labelsize=11
        )

# AXIS LABELS

for ax in axes[1]:

    ax.set_xlabel(
        'Depth Difference (ft)\n'
        '(NSI - Philadelphia)',
        fontsize=12
    )

for ax in axes[0]:

    ax.set_xlabel('')

# REMOVE DUPLICATE LEGENDS

for ax in axes.ravel():

    if ax.get_legend() is not None:

        ax.get_legend().remove()

# SHARED LEGEND

legend_elements = [

    Patch(
        facecolor=palette['Basement'],
        label='Basement'
    ),

    Patch(
        facecolor=palette['No Basement'],
        label='No Basement'
    )

]

fig.legend(
    handles=legend_elements,
    fontsize=10,
    loc='upper center',
    bbox_to_anchor=(0.8, .88),
    ncol=1,
    frameon=False
)
fig.savefig(
    join(
        FIG_DIR_SUPP,
        'depth_mismatch_discrepancies.png'
    ),
    dpi=300,
    bbox_inches='tight'
)

In [ ]:
depth_discrep_nsi = matched_df[matched_df['parcel_number'].isin(plot_sub['comp_id'])]['fd_id']
nsi_clip_out[nsi_clip_out['fd_id'].isin(depth_discrep_nsi)].groupby(['source', 'ftprntsrc']).size()

#### Plot some of the depth discrepancies to show how building footprint data can still be inaccurate

In [ ]:
def plot_matched_property(
    parcel_number,
    buffer_dist=40,  # meters
    filename='matched_property.png'
):

    PLOT_CRS = 3857

    # --------------------------------------------------
    # MATCH LOOKUP
    # --------------------------------------------------

    match_sub = (
        matched_df_main[
            matched_df_main['parcel_number']
            == parcel_number
        ]
        .copy()
    )

    if len(match_sub) == 0:

        raise ValueError(
            f'Parcel {parcel_number} not found.'
        )

    fd_ids = (
        match_sub['fd_id']
        .dropna()
        .unique()
    )

    bfids = (
        match_sub['bfid']
        .dropna()
        .unique()
    )

    # --------------------------------------------------
    # TARGET FEATURES
    # --------------------------------------------------

    parcel_sub = (
        parcel[
            parcel['BRT_ID']
            == parcel_number
        ]
        .to_crs(PLOT_CRS)
    )

    if len(parcel_sub) == 0:

        raise ValueError(
            f'Parcel {parcel_number} not found in parcel layer.'
        )

    phil_target = (
        phil_inv_out[
            phil_inv_out['bfid'].isin(
                bfids
            )
        ]
        .to_crs(PLOT_CRS)
    )

    nsi_target = (
        nsi_clip_out[
            nsi_clip_out['fd_id'].isin(
                fd_ids
            )
        ]
        .to_crs(PLOT_CRS)
    )

    # --------------------------------------------------
    # EXTENT FROM STRUCTURES
    # --------------------------------------------------

    geoms = []

    if len(phil_target) > 0:

        geoms.extend(
            phil_target.geometry.tolist()
        )

    if len(nsi_target) > 0:

        geoms.extend(
            nsi_target.buffer(20).geometry.tolist()
        )

    if len(geoms) == 0:

        geoms.extend(
            parcel_sub.geometry.tolist()
        )

    geom = (
        gpd.GeoSeries(
            geoms,
            crs=PLOT_CRS
        )
        .union_all()
        .buffer(buffer_dist)
    )

    clip_box = gpd.GeoDataFrame(
        geometry=[geom],
        crs=PLOT_CRS
    )

    xmin, ymin, xmax, ymax = (
        clip_box.total_bounds
    )

    # --------------------------------------------------
    # FIGURE
    # --------------------------------------------------

    fig, ax = plt.subplots(
        figsize=(6, 6),
        dpi=300
    )

    # --------------------------------------------------
    # BASEMAP
    # --------------------------------------------------

    ax.set_xlim(
        xmin,
        xmax
    )

    ax.set_ylim(
        ymin,
        ymax
    )

    cx.add_basemap(
        ax,
        source=cx.providers.Esri.WorldImagery,
        attribution_size=4
    )

    # --------------------------------------------------
    # PARCEL OUTLINE
    # --------------------------------------------------

    parcel_sub.plot(
        ax=ax,
        facecolor='none',
        edgecolor='gray',
        linewidth=1,
        zorder=10
    )

    # --------------------------------------------------
    # PHILADELPHIA FOOTPRINT
    # --------------------------------------------------

    phil_target.plot(
        ax=ax,
        facecolor='#0077BB',
        edgecolor='#0077BB',
        alpha=.5,
        linewidth=1,
        zorder=11
    )

    # --------------------------------------------------
    # NSI POINT
    # --------------------------------------------------

    nsi_target.plot(
        ax=ax,
        color='red',
        markersize=40,
        zorder=12
    )

    # --------------------------------------------------
    # AXES
    # --------------------------------------------------

    ax.set_title('')

    ax.set_xticks([])
    ax.set_yticks([])

    ax.set_xlabel('')
    ax.set_ylabel('')

    # --------------------------------------------------
    # LEGEND
    # --------------------------------------------------

    legend_elements = [

        Patch(
            facecolor='none',
            edgecolor='gray',
            label='Parcel'
        ),

        Patch(
            facecolor='#0077BB',
            edgecolor='#0077BB',
            alpha=.5,
            label='Philadelphia Structure'
        ),

        Line2D(
            [0],
            [0],
            marker='o',
            linestyle='',
            color='red',
            markerfacecolor='red',
            markersize=8,
            label='NSI Structure'
        )

    ]

    ax.legend(
        handles=legend_elements,
        fontsize=10,
        frameon=True,
        loc='upper right'
    )

    # --------------------------------------------------
    # SAVE
    # --------------------------------------------------

    fig.savefig(
        join(
            FIG_DIR,
            'supp',
            filename
        ),
        dpi=300,
        bbox_inches='tight'
    )

    return fig, ax

In [ ]:
plot_matched_property(
    '381114705',
    buffer_dist=100,
    filename='largest_nsi_depth_over.png'
)

In [ ]:
plot_matched_property(
    '442288000',
    buffer_dist=100,
    filename='largest_nsi_depth_under.png'
)

The example below shows 7 NSI structures to 1 Philly structure. It's a complicated Philly apartment complex. 6 of the NSI structures say RES1. 1 says RES3 but with basement (it's a no basement property). It underestimates sq ft by an order of magnitude. (NSI says 14k, assessment says 132k). Value is off by factor of 2. 

In [ ]:
plot_matched_property(
    '886652600',
    buffer_dist=100,
    filename='many_random_nsi_points.png'
)

The example below shows 31 NSI records to one Philly building - it's recorded as an apartment. NSI says these ares RES3 but inconsistent characteristic assignments. 
1SWB    12
1SNB     9
2SWB     7
2SNB     2
It's a 3SNB (2SNB for ddf purposes). 
It overestimates sq ft a little bit (68k vs 56k). It apparently is identifying some structures based on the MBL data that aren't unique footprints. 
It overestimates value. 6612305 vs. 8346380 (factor of 1.7)

In [ ]:
plot_matched_property(
    '881124500',
    buffer_dist=100,
    filename='many_nsi_one_phil_example.png'
)

Below is an illustrative condo example. There are 17 NSI records here. NSI says these are all RES1, but they are technically RES3. This is also recorded as RES CONDO 4STY MASONRY in the Philadelphia assessor data. Therefore, it does not end up in the loss inventory. The aggregated value of units from Philly inventory is 9471600. From NSI: 5073138. So, NSI ~ .5 the value, potentially underestimating sq ft. Again, all parcel source and ftprnt source from MBL. These are very difficult entries for NSI to handle since there will be multiple distinct parcel records for condos but usually only one spatially referenced one. There needs to be some address aggregation of parcel information. As for footprint, it could be more accurate to use parcel boundaries as an overlay for any identified footprint and treat continguous footprints as a single structure. 

In [ ]:
plot_matched_property(
    '888101711',
    buffer_dist=100,
    filename='condo_example.png'
)

### Discrepancy decomposition

In [ ]:
anova_df = parcel_gb[
    parcel_gb['mismatch_plot'] != 'Multiple Buildings\nOn Parcel'
].copy()

# Location mismatch
# True when one inventory contains a structure
# and the other does not
# These are NSI/Philly Only, as well as cases
# where there is location discrepancy in depths
anova_df['location_mismatch'] = (
    (anova_df['mismatch_plot']
    .isin(
        ['NSI Only', 'Philly Only']
    )) |
    (anova_df['009_nsi'] != anova_df['009_phil'])
)

# Characteristic mismatch
# Match      -> DDF Match
# Mismatch   -> DDF Mismatch
# None       -> No structure comparison possible
anova_df['characteristic_mismatch'] = np.select(
    [
        anova_df['mismatch_plot'] == 'DDF Match',
        anova_df['mismatch_plot'] == 'DDF Mismatch'
    ],
    [
        'Match',
        'Mismatch'
    ],
    default='None'
)

model_rel = ols(
    '''
    rel_loss_diff
    ~ location_mismatch * C(depth_bins) * C(characteristic_mismatch)
    ''',
    data=anova_df
).fit()

model_loss = ols(
    '''
    loss_diff
    ~ location_mismatch * C(depth_bins) * C(characteristic_mismatch)
    + value_diff
    ''',
    data=anova_df
).fit()

anova_rel = anova_lm(
    model_rel,
    typ=2
)

anova_loss = anova_lm(
    model_loss,
    typ=2
)

def add_eta_sq(tbl):

    tbl = tbl.copy()

    ss_total = tbl['sum_sq'].sum()

    tbl['eta_sq'] = (
        tbl['sum_sq']
        /
        ss_total
    )

    return tbl

print('\n' + '='*80)
print('REL LOSS ANOVA')
print('\n' + '='*80)
print(add_eta_sq(anova_rel))

print('\n' + '='*80)
print('ABSOLUTE LOSS ANOVA')
print('\n' + '='*80)
print(add_eta_sq(anova_loss))

### More detailed discrepancies
Similar to Figure 3 but only for those with location matches and with unique characteristic matches broken out. Since DDF Match and DDF Mismatch are very difficult to see in Fig 3, esp. at smaller depths, this helps characterize the discrepancies in more detail. 

In [ ]:
bfid_sub = phil_inv_ens[
    (phil_inv_ens['num_story'] == 2) &
    (phil_inv_ens['found_type'] == 'B')
].index

comp_ids_sub = (
    match_analysis[
        match_analysis['bfid'].isin(bfid_sub)
    ]['comp_id']
    .unique()
)

parcel_sub = (
    parcel_gb[
        parcel_gb['comp_id'].isin(comp_ids_sub)
    ]
    .copy()
)

plot_sub = parcel_sub.copy()

plot_sub['ddf_match'] = (
    plot_sub['mismatch_reason']
)

plot_sub.loc[
    plot_sub['ddf_match'] == 'None',
    'ddf_match'
] = 'Both 2SWB'

plot_sub['value_diff_thou'] = (
    plot_sub['value_diff']
    / 1000
)

plot_sub['loss_diff_thou'] = (
    plot_sub['loss_diff']
    / 1000
)

order = [
    'Both 2SWB',
    'NSI: 2SNB\nPhilly: 2SWB',
    'NSI: 1SWB\nPhilly: 2SWB',
    'NSI: 1SNB\nPhilly: 2SWB'
]

fig, ax = plt.subplots(
    figsize=(6, 8),
    nrows=5,
    sharex=True,
    dpi=300
)

# RELATIVE LOSS DIFFERENCE

ax[0].axhline(
    0,
    lw=2,
    color='lightgray',
    zorder=0
)

sns.pointplot(
    data=plot_sub,
    x='ddf_match',
    y='rel_loss_diff',
    order=order,
    errorbar='sd',
    linestyles='none',
    capsize=.2,
    ax=ax[0]
)

# VALUE DIFFERENCE

ax[1].axhline(
    0,
    lw=2,
    color='lightgray',
    zorder=0
)

sns.pointplot(
    data=plot_sub,
    x='ddf_match',
    y='value_diff_thou',
    order=order,
    errorbar='sd',
    linestyles='none',
    capsize=.2,
    ax=ax[1]
)

# DAMAGE DIFFERENCE

ax[2].axhline(
    0,
    lw=2,
    color='lightgray',
    zorder=0
)

sns.pointplot(
    data=plot_sub,
    x='ddf_match',
    y='loss_diff_thou',
    order=order,
    errorbar='sd',
    linestyles='none',
    capsize=.2,
    ax=ax[2]
)

# COUNTS

sns.countplot(
    data=plot_sub,
    x='ddf_match',
    order=order,
    ax=ax[3]
)

# TOTAL DAMAGE CONTRIBUTION

ax[4].axhline(
    0,
    lw=2,
    color='lightgray',
    zorder=0
)

plot_sub['loss_diff_mil'] = (
    plot_sub['loss_diff']
    / 1e6
)

sns.barplot(
    data=plot_sub,
    x='ddf_match',
    y='loss_diff_mil',
    estimator='sum',
    errorbar=None,
    order=order,
    ax=ax[4]
)

for container in ax[4].containers:
    ax[4].bar_label(
        container,
        fmt='%.1f',
        padding=3,
        fontsize=11
    )
ax[4].set_ylim(
    -22,
    2
)

ax[1].set_yscale('symlog')
ax[2].set_yscale('symlog')

ax[1].set_yticks(
    [-1000, -100, -10, 10, 100, 1000]
)

ax[2].set_yticks(
    [-100, -10, 10, 1000]
)

ax[1].set_yticklabels(
    ['-1000', '-100', '-10', '10', '100', '1000']
)

ax[2].set_yticklabels(
    ['-100', '-10', '10' ,'100']
)

# LABELS

ax[0].set_ylabel(
    'Rel.Loss Diff.',
    size=12
)

ax[1].set_ylabel(
    'Value Diff. ($K)',
    size=12
)

ax[2].set_ylabel(
    'Damage Diff. ($K)',
    size=12
)

ax[3].set_ylabel(
    'Count',
    size=12
)

ax[4].set_ylabel(
    'Total Discrep. ($M)',
    size=12
)

ax[4].set_xlabel(
    'Depth-Damage Function (Mis)Matches',
    size=12
)

fig.align_ylabels(ax)

for axis in ax:

    axis.tick_params(
        labelsize=12
    )

fig.tight_layout()

fig.savefig(join(FIG_DIR_SUPP, 'discrep_2swb.png'), bbox_inches='tight', dpi=300)


In [ ]:
bfid_sub = phil_inv_ens[
    (phil_inv_ens['num_story'] == 2) &
    (phil_inv_ens['found_type'] == 'S')
].index

comp_ids_sub = (
    match_analysis[
        match_analysis['bfid'].isin(bfid_sub)
    ]['comp_id']
    .unique()
)

parcel_sub = (
    parcel_gb[
        parcel_gb['comp_id'].isin(comp_ids_sub)
    ]
    .copy()
)

plot_sub = parcel_sub.copy()

plot_sub['ddf_match'] = (
    plot_sub['mismatch_reason']
)

plot_sub.loc[
    plot_sub['ddf_match'] == 'None',
    'ddf_match'
] = 'Both 2SNB'

plot_sub['value_diff_thou'] = (
    plot_sub['value_diff']
    / 1000
)

plot_sub['loss_diff_thou'] = (
    plot_sub['loss_diff']
    / 1000
)

order = [
    'Both 2SNB',
    'NSI: 2SWB\nPhilly: 2SNB',
    'NSI: 1SWB\nPhilly: 2SNB',
    'NSI: 1SNB\nPhilly: 2SNB'
]

fig, ax = plt.subplots(
    figsize=(6, 8),
    nrows=5,
    sharex=True,
    dpi=300
)

# RELATIVE LOSS DIFFERENCE

ax[0].axhline(
    0,
    lw=2,
    color='lightgray',
    zorder=0
)

sns.pointplot(
    data=plot_sub,
    x='ddf_match',
    y='rel_loss_diff',
    order=order,
    errorbar='sd',
    linestyles='none',
    capsize=.2,
    ax=ax[0]
)

# VALUE DIFFERENCE

ax[1].axhline(
    0,
    lw=2,
    color='lightgray',
    zorder=0
)

sns.pointplot(
    data=plot_sub,
    x='ddf_match',
    y='value_diff_thou',
    order=order,
    errorbar='sd',
    linestyles='none',
    capsize=.2,
    ax=ax[1]
)

# DAMAGE DIFFERENCE

ax[2].axhline(
    0,
    lw=2,
    color='lightgray',
    zorder=0
)

sns.pointplot(
    data=plot_sub,
    x='ddf_match',
    y='loss_diff_thou',
    order=order,
    errorbar='sd',
    linestyles='none',
    capsize=.2,
    ax=ax[2]
)

# COUNTS

sns.countplot(
    data=plot_sub,
    x='ddf_match',
    order=order,
    ax=ax[3]
)

# TOTAL DAMAGE CONTRIBUTION

ax[4].axhline(
    0,
    lw=2,
    color='lightgray',
    zorder=0
)

plot_sub['loss_diff_mil'] = (
    plot_sub['loss_diff']
    / 1e6
)

sns.barplot(
    data=plot_sub,
    x='ddf_match',
    y='loss_diff_mil',
    estimator='sum',
    errorbar=None,
    order=order,
    ax=ax[4]
)

for container in ax[4].containers:
    ax[4].bar_label(
        container,
        fmt='%.1f',
        padding=1,
        fontsize=11
    )
ax[4].set_ylim(
    0,
    16.5
)

ax[1].set_yscale('symlog')
ax[2].set_yscale('symlog')

ax[1].set_yticks(
    [-1000, -100, -10, 10, 100, 1000]
)

ax[2].set_yticks(
    [-100, -10, 10, 1000]
)

ax[1].set_yticklabels(
    ['-1000', '-100', '-10', '10', '100', '1000']
)

ax[2].set_yticklabels(
    ['-100', '-10', '10' ,'100']
)

# LABELS

ax[0].set_ylabel(
    'Rel.Loss Diff.',
    size=12
)

ax[1].set_ylabel(
    'Value Diff. ($K)',
    size=12
)

ax[2].set_ylabel(
    'Damage Diff. ($K)',
    size=12
)

ax[3].set_ylabel(
    'Count',
    size=12
)

ax[4].set_ylabel(
    'Total Discrep. ($M)',
    size=12
)

ax[4].set_xlabel(
    'Depth-Damage Function (Mis)Matches',
    size=12
)

fig.align_ylabels(ax)

for axis in ax:

    axis.tick_params(
        labelsize=12
    )

fig.tight_layout()

fig.savefig(join(FIG_DIR_SUPP, 'discrep_2snb.png'), bbox_inches='tight', dpi=300)

### Fig S2
Comparison of damage estimates for matched structures at different depths. The rows show a different matched structure (i.e., structures with the same location, number of stories, and basement type) across inventories. From left to right, the columns show percent damage to a structure, structure value, and structure damage. Blue histograms represent the Philadelphia ensemble, which accounts for uncertainty in both first-floor elevation and depth-damage functions (DDF). Red vertical lines indicate deterministic National Structure Inventory (NSI) estimates, which do not account for DDF uncertainty. Orange histograms represent the NSI inventory accounting for DDF uncertainty but not in first-floor elevation. Box plots above each histogram summarize the distributions of the ensemble members, with diamonds denoting ensemble means. 

In [ ]:
def plot_building_damage_comparison(phil_data, nsi_data, ens_comp, bfids,
                                   depth_col='depth_ft',
                                   n_bins=50,
                                   dam_cols=['rel_loss', 'val_s', 'loss'],
                                   dam_labels=['Damage (%)', 'Value ($ Thousands)', 'Damage ($ Thousands)'],
                                   scale_factors=[1, 1e3, 1e3],
                                   phil_label='Philly Ensemble',
                                   ens_label='Alternate Ensemble'):
    """
    Create a generalized comparison of building damages between Philadelphia and NSI data.
    
    Parameters:
    -----------
    phil_data : DataFrame
        Philadelphia building data with damage columns
    nsi_data : DataFrame
        NSI building data with damage columns
    ens_comp : DataFrame
        Second ensemble data to compare with Philadelphia
    bfids : list
        List of building IDs to analyze
    depth_col : str
        Name of the column containing depth information
    n_bins: int
        Number of bins for histogram
    dam_cols : list
        List of damage column names to compare
    dam_labels : list
        Labels for the damage columns
    scale_factors : list
        Scaling factors for the damage values
    phil_label : str
        Label for the Philadelphia ensemble
    ens_label : str
        Label for the second ensemble
    """
    # Create figure with a grid layout
    fig = plt.figure(figsize=(10, 10), dpi=300)
    
    # Define colors for consistency
    philly_color = sns.color_palette('Set1')[1]
    ens_color = sns.color_palette('Set2')[1]
    nsi_color = 'red'
    
    # Process each building
    for i, bfid in enumerate(bfids):
        # Filter data for this building
        phil_plot = phil_data[phil_data['bfid'] == bfid]
        nsi_plot = nsi_data[nsi_data['bfid'] == bfid]
        ens_plot = ens_comp[ens_comp['bfid'] == bfid]
        
        # Get depth directly from the data
        depth = phil_plot[depth_col].iloc[0]  # Use Philadelphia data for depth
        
        # Process each damage column
        for j, (dam_col, dam_label, scale_factor) in enumerate(zip(dam_cols, dam_labels, scale_factors)):
            # Calculate subplot position
            subplot_idx = i * len(dam_cols) + j + 1
            
            # Create a subplot with 2 rows (boxplot on top, histogram below)
            ax = plt.subplot(len(bfids), len(dam_cols), subplot_idx)
            
            # Create a gridspec for this subplot to have boxplot on top and histogram below
            gs = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=ax.get_subplotspec(),
                                                 height_ratios=[1, 3], hspace=0)
            
            # Create the boxplot axes (top) and histogram axes (bottom)
            ax_box = fig.add_subplot(gs[0])
            ax_hist = fig.add_subplot(gs[1], sharex=ax_box)
            
            # Hide the main axes
            ax.axis('off')
            
            # Prepare data from both ensembles
            phil_values = (phil_plot.groupby(['sow_ind'])[[dam_col]].sum()/scale_factor).reset_index()[dam_col]
            ens_values = (ens_plot.groupby(['sow_ind'])[[dam_col]].sum()/scale_factor).reset_index()[dam_col]
            
            # Create combined DataFrame for seaborn
            if dam_col != 'val_s':
                combined_data = pd.DataFrame({
                    'value': pd.concat([phil_values, ens_values]),
                    'source': [phil_label] * len(phil_values) + [ens_label] * len(ens_values)
                })
            else:
                combined_data = pd.DataFrame({
                    'value': phil_values,
                    'source': [phil_label] * len(phil_values)
                })
            
            # Create histograms using seaborn
            sns.histplot(data=combined_data, x='value', hue='source', 
                        bins=n_bins, alpha=0.75, ax=ax_hist,
                        palette={phil_label: philly_color, ens_label: ens_color},
                        legend=False,
                        element="step", fill=True, stat="count")
            
            # Create boxplots using seaborn
            sns.boxplot(data=combined_data, x='value', y='source', 
                       orient='h', ax=ax_box,
                       hue='source',
                       legend=False,
                       palette={phil_label: philly_color, ens_label: ens_color},
                       showmeans=True,
                       meanprops={'markerfacecolor': 'firebrick',
                                 'markeredgecolor': 'black',
                                 'marker': 'D'})
            
            # Remove y-axis labels but keep the ticks for visual separation
            ax_box.set_yticklabels([])
            ax_box.set_yticks([])
            ax_box.set_xlabel('')
            ax_box.axis('off')  # Hide the boxplot axes
            
            # Add NSI reference line
            nsi_value = nsi_plot[dam_col].sum()/scale_factor
            ax_hist.axvline(nsi_value, color=nsi_color, linestyle='-', linewidth=2)
            ax_box.axvline(nsi_value, color=nsi_color, linestyle='-', linewidth=2)
            
            # Configure histogram
            ax_hist.grid(False)
            if i == len(bfids) - 1:  # Only add x-label to bottom row
                ax_hist.set_xlabel(dam_label, size=16)
            else:
                ax_hist.set_xlabel('')
            
            if j == 0 and i == 1:  # Only add y-label to first column of first row
                ax_hist.set_ylabel('Number of Ensemble Members', size=16)
            else:
                ax_hist.set_ylabel('')
            
            ax_hist.tick_params(labelsize=12)
            ax_hist.xaxis.set_major_locator(plt.MaxNLocator(5))
            
            # Process the x ticks for % damage
            if j == 0:
                ax_hist.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, symbol=None, decimals=0))
            
            # Remove the automatically generated legend
            if ax_hist.get_legend():
                ax_hist.get_legend().remove()

            # Add row title for depth
            # if j == 0:
                # ax.annotate(f"Matched Structure With Inundation of {depth:.2f} ft",
                #            xy=(1.1, 1.05),
                #            horizontalalignment='center',
                #            xycoords='axes fraction',
                #            fontsize=16)
            if j == 1:
                
                ax.set_title(f"Fully Matched Structure With Inundation of {depth:.2f} ft",
                             size=14)

    # Create legend elements
    legend_elements = [
        Patch(facecolor=philly_color, alpha=.75, label=phil_label),
        Patch(facecolor=ens_color, alpha=.75, label=ens_label),
        Line2D([0], [0], color=nsi_color, lw=2, label='NSI w/o DDF Uncertainty'),
        Line2D([0], [0], marker='D', markerfacecolor='firebrick',
              label='Ensemble Mean', ls='', markeredgecolor='black', markersize=8)
    ]
    
    # Adjust layout
    plt.tight_layout()
    plt.subplots_adjust(wspace=.22, hspace=None)
    
    # Add the legend
    plt.legend(handles=legend_elements,
              fontsize='x-large',
              loc='center',
              bbox_to_anchor=(-.8, -.75),
              frameon=True,
              ncol=2
    )
    
    return fig

# Find bfids on cases you'd like to illustrate
bfids = [22461 , 42713 , 29582]

fd_bfid_lnk = (
    matched_df_main
    .dropna(
        subset=['fd_id', 'bfid']
    )
    .drop_duplicates('fd_id')
    .set_index('fd_id')['bfid']
    .to_dict()
)

nsi_bench = all_results['no_unc']
nsi_bench = nsi_inv_ens.join(nsi_bench).reset_index()
nsi_bench['rel_loss'] = nsi_bench[dam_col]/nsi_bench['val_struct']
nsi_bench['val_s'] = nsi_bench['val_struct'].copy()
nsi_bench['bfid'] = nsi_bench['fd_id'].map(fd_bfid_lnk)
nsi_ddf_unc['bfid'] = nsi_ddf_unc['fd_id'].map(fd_bfid_lnk)
nsi_ddf_unc[dg_id] = nsi_ddf_unc['fd_id'].map(nsi_depths_df[dg_id])*3.28084
main_phil[dg_id] = main_phil['bfid'].map(phil_depths_df[dg_id])*3.28084
main_phil['rel_loss'] = main_phil[dam_col] / main_phil['val_s']
nsi_ddf_unc['rel_loss'] = nsi_ddf_unc[dam_col] / nsi_ddf_unc['val_s']


fig = plot_building_damage_comparison(
    phil_data=main_phil, 
    ens_comp=nsi_ddf_unc,
    nsi_data=nsi_bench, 
    bfids=bfids,
    depth_col=dg_id,
    dam_cols=['rel_loss', 'val_s', dam_col],
    dam_labels= ['Damage (%)', 'Value ($ Thousands)', 'Damage ($ Thousands)'],
    scale_factors=[1, 1000, 1000],
    ens_label='NSI w/ DDF Uncertainty Only'
)

fig.savefig(join(FIG_DIR_SUPP, 'match_dam_var.png'), bbox_inches='tight', dpi=300)


### Major sensitivity analyses on decision relevant metrics
Redo figure 5 for each of the sensitivity analyses

First we look at a value at risk adjustment for 3 story buildings. 

In [ ]:
exps = ['nsi_nounc', 'nsi_ddfs', 'nsi_unsafe', 'nsi_phil', 'nsi_allphil', 'nsiadj_unsafe']
titles = ['Status Quo: Standard NSI application',
          'Main Comparison of Study: Sample damage uncertainty',
          'NSI + Enhanced Uncertainty: Sample structure uncertainty',
          'Refinement 1 (R1): Adjust for local characteristics',
          'R2: R1 + Rough structure value calibration',
          'R3: Use local footprints and structure values']

# Define distinct colors for each method
hues = ['#6EA6CD', '#98CAE1', '#C2E4EF', '#FDB366', '#F67E4B', '#DD3D2D']

# Create the plot
metric_ens = pd.read_parquet(join(FO, 'ensembles', 'all_metrics_sa1.pqt'))
fig, axes = plot_skill_metrics_box(metric_ens, exps, titles, hues)
fig.savefig(join(FIG_DIR_SUPP, 'fig5_sa1.png'), bbox_inches='tight', dpi=300)

In [ ]:
phil_inv_ens['stories_n'].value_counts()

In [ ]:
nsi_inv_ens['stories_n'].value_counts()

There are roughly 4x as many 3 story buildings in Philly inventory, resulting in a larger reduction in value at risk for these properties. It follows that the total loss estimate for Philly will go down, so the total discrepancy goes up. The structure value adjustment isn't as effective as it was before since the characteristics are off and the value at risk adjustment doesn't also accompany. There are otherwise similar patterns on RMSE & misclassification for various inventory approaches, which makes sense since there are 3S buildings throughout the city. If there was a concentration of tracts with or without, and these happened to have many exposed properties, this adjustment could have had a lager impact on the decision relevant metrics

Next we look at the consequences of using RES3 DDFs when an inventory says that's the occupancy type

In [ ]:
exps = ['nsi_nounc', 'nsi_ddfs', 'nsi_unsafe', 'nsi_phil', 'nsi_allphil', 'nsiadj_unsafe']
titles = ['Status Quo: Standard NSI application',
          'Main Comparison of Study: Sample damage uncertainty',
          'NSI + Enhanced Uncertainty: Sample structure uncertainty',
          'Refinement 1 (R1): Adjust for local characteristics',
          'R2: R1 + Rough structure value calibration',
          'R3: Use local footprints and structure values']
# Define distinct colors for each method
hues = ['#6EA6CD', '#98CAE1', '#C2E4EF', '#FDB366', '#F67E4B', '#DD3D2D']

# Create the plot
metric_ens = pd.read_parquet(join(FO, 'ensembles', 'all_metrics_sa2.pqt'))
fig, axes = plot_skill_metrics_box(metric_ens, exps, titles, hues)
fig.savefig(join(FIG_DIR_SUPP, 'fig5_sa2.png'), bbox_inches='tight', dpi=300)

This is very similar looking to the main result. Even though nearly every Philadelphia property ends up updating its damage functions, this has a modest effect, presumably because most exposure is at very small depths and damages at these depths are very similar across DDFs. Even at lager depths, the difference in damage isn't so big relative to the dominating influence of NSI only vs. Philly only and where those happen to be (spatially, which is what matters for the bottom 2 metrics)

Finally we look at increasing the lowest depth thresold for damage

In [ ]:
exps = ['nsi_nounc', 'nsi_ddfs', 'nsi_unsafe', 'nsi_phil', 'nsi_allphil', 'nsiadj_unsafe']
titles = ['Status Quo: Standard NSI application',
          'Main Comparison of Study: Sample damage uncertainty',
          'NSI + Enhanced Uncertainty: Sample structure uncertainty',
          'Refinement 1 (R1): Adjust for local characteristics',
          'R2: R1 + Rough structure value calibration',
          'R3: Use local footprints and structure values']

# Define distinct colors for each method
hues = ['#6EA6CD', '#98CAE1', '#C2E4EF', '#FDB366', '#F67E4B', '#DD3D2D']

# Create the plot
metric_ens = pd.read_parquet(join(FO, 'ensembles', 'all_metrics_sa3.pqt'))
fig, axes = plot_skill_metrics_box(metric_ens, exps, titles, hues)
fig.savefig(join(FIG_DIR_SUPP, 'fig5_sa3.png'), bbox_inches='tight', dpi=300)

These results are relatively similar to the main. Removing low depth properties removes a lage number of damaged buildings but there overall discrepancy is small since loss estimates are small. The total discrepancy % goes up slightly. The Philly Only proportion in this subset is slightly larger than overall, and the other discrepancies end up being slightly positive. As a result, % discrepancy goes up. This seems random and not like it will always be the case. 